# Importing Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import confusion_matrix
import math
import joblib
from sklearn.metrics import mean_squared_error, mean_pinball_loss, r2_score, classification_report, confusion_matrix, mean_absolute_percentage_error
from imblearn.over_sampling import SMOTENC
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam
import time
from tensorflow.keras.utils import Sequence
from sklearn.preprocessing import OneHotEncoder
from itertools import permutations
import random
from IPython.display import display, HTML
from IPython.display import Markdown, display
# from keras.saving import register_keras_serializable # Original import
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras.preprocessing.sequence import pad_sequences
from math import ceil
import datetime
import gc
%matplotlib inline

In [ ]:
# Display all columns
pd.set_option('display.max_columns', None)

# Mounting to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

drive_path = '/content/drive/MyDrive/system_metrics_predictions/'

Mounted at /content/drive


In [ ]:
# Import Projection Methods
import sys
sys.path.append(drive_path+"notebooks/")
from utils.domain_projection import *

In [ ]:
from importlib import reload
import utils.domain_projection

reload(utils.domain_projection)

<module 'utils.domain_projection' from '/content/drive/MyDrive/system_metrics_predictions/notebooks/refactor_code/utils/domain_projection.py'>

# Constants

In [ ]:
VERSION = 'V3'
PROJECTION = 'adapter'
INSERT_SOURCE = 'dropout_final'
LEVEL = "data_reduction_experiment"
STEP = 40
MODE = "multiple"

PATH_TO_SPLITS = drive_path + f"data/splits/V2/"

PATH_TO_SCALERS = drive_path + f'output/scalers/multi_task_domain_adaptation/{LEVEL}/{VERSION}/'
PATH_TO_LOAD_MODELS = drive_path + 'output/system-forecaster-models/baselines/V1/'
PATH_TO_MODELS = drive_path + f"output/system-forecaster-models/multi_task_domain_adaptation/{LEVEL}/{VERSION}/"
PATH_TO_RESULTS = drive_path + f"output/results/multi_task_domain_adaptation/{LEVEL}/{VERSION}/"
PATH_TO_TRAINING_LOGS = drive_path + f"output/training_logs/multi_task_domain_adaptation/{LEVEL}/{VERSION}/"
PATH_TO_PLOTS  = drive_path + f"output/plots/multi_task_domain_adaptation/{LEVEL}/{VERSION}/"

NODE_TYPES = ["LIGHT", "MID", "HEAVY"]
features = ['node_type',
 'rate_function_curl',
 'rate_function_eat_memory',
 'rate_function_env',
 'rate_function_figlet',
 'rate_function_nmap',
 'rate_function_shasum']

GB = 1024 ** 3  # 1 GB in bytes

# Lookup for node capacities
CAPACITIES = {
    0: 24 * GB,   # Heavy
    1: 16 * GB,   # Mid
    2:  8 * GB    # Light
}

MODEL_NAME = f"system_forecaster_functions_all_multi_task_domain_adaptation_{VERSION}"

# Training Time Dataframe

In [ ]:
# Define file path
training_time_path = drive_path +"Training_Time.csv"

# Check if the CSV file for storing the training time of models exists
if os.path.exists(training_time_path):
    time_df = pd.read_csv(training_time_path,index_col=0)
    print("CSV file loaded successfully.")
else:
    time_df = pd.DataFrame()  # Create an empty DataFrame
    print("CSV file does not exist. Created a new DataFrame.")

# Check if the column_names column exists, else add it
if MODEL_NAME not in time_df.columns:
    time_df[MODEL_NAME] = None  # You can set a default value like None or 0
    print(f"Column '{MODEL_NAME}' added to the DataFrame.")

CSV file loaded successfully.


# Utility Functions




### Data and Cleaning

In [ ]:
# Function to import data
def import_data(path):
  df = pd.DataFrame()
  for node_type in NODE_TYPES:
      # Retrieve all files in the data folder
      file_csv = [file for file in os.listdir(path + node_type) if file.endswith('.csv')]
      # Create the dataframe by concatenating all read files
      dataframes = []
      for file in file_csv:
          file_path = os.path.join(path + node_type, file)
          df_temp = pd.read_csv(file_path)
          # Remove the columns in the dataframe that begin with "function_"
          df_temp.drop(columns=[col for col in df_temp if col.startswith('function_')], inplace=True)
          # Add the column "node_type" and assign the value of 'type' to all rows
          if node_type == "HEAVY":
              df_temp["node_type"] = 0
          elif node_type == "MID":
              df_temp["node_type"] = 1
          else:
              df_temp["node_type"] = 2

          dataframes.append(df_temp)

      df = pd.concat([df, *dataframes], axis=0, ignore_index=True)

  return df

In [ ]:
# Function used to fill NaN values within the dataframe X
def fill_NaN(X):
  for col in X:
    if(col.startswith('success_rate_')):
      X.loc[:, col] = X.loc[:, col].fillna(1)
    else:
      X.loc[:, col] = X.loc[:, col].fillna(0)
  return X

In [ ]:
# Function to assign zero to noise metrics where requests rates are 0
def assign_zero_to_noise(df):
  functions = [col[14:] for col in df if col.startswith('rate')]

  for function in functions:
      df.loc[df['rate_function_' + function] == 0, ['cpu_usage_function_' + function, 'ram_usage_function_' + function, 'power_usage_function_' + function, 'replica_' + function]] = 0
  return df

In [ ]:
# Function to select relevant columns
def select_columns(df):
  targets = [col for col in df if (col.startswith('cpu_usage_') or
                                   col.startswith('ram_usage_') or
                                   col.startswith('overloaded_node')
                                   or col.startswith('medium_latency')) and 'idle' not in col]# or col.startswith('replica')
  features = [col for col in df if col.startswith('rate_') or 'node_type' in col]
  features.sort()
  df = df[features+targets]
  return df

In [ ]:
# Function to Rename Eat Memory column's Name
def rename_eat_memory(df):
  new_column_name = {i:i.replace("-","_") for i in df.keys() if "eat" in i}
  df.rename(columns=new_column_name, inplace=True)
  return df

In [ ]:
#Remove rows which are overloaded (overloaded_node == 1).

def remove_overloaded_rows(df):
    df = df.copy()
    return df[df["overloaded_node"] == 0]

In [ ]:
#Remove rows where all specified columns have value 0.

def remove_baselines(df,features):
    df = df.copy()
    return df[~(df[features] == 0).all(axis=1)]

In [ ]:
# 0 = all 0, 1 = all 1, 2 = mix
def group_status(x):
    if (x == 0).all():
        return 0
    elif (x == 1).all():
        return 1
    else:
        return 2

In [ ]:
# Function to add overload status and overload ratio.
def overload_status_ratio(df,featues,target):
  grouped = df.groupby(features)

  df['overloaded_status'] = grouped[target].transform(group_status)

  # Ratio of 1’s in the group
  df['overloaded_ratio'] = grouped[target].transform('mean')

  return df


In [ ]:
def ram_usage_to_percentage(ram_usage, node_type):
    """Return RAM utilization % given usage (bytes) and node type."""
    return (ram_usage / CAPACITIES[node_type]) * 100

In [ ]:
def compute_ram_usage_percentage_theoretical(df, ram_col="ram_usage_node", node_col="node_type"):
    """Apply ram_usage_to_perc on the DataFrame."""
    # df["ram_usage_node_perc_theor"] = df.apply(
    #     lambda row: ram_usage_to_percentage(row[ram_col], row[node_col]),
    #     axis=1
    # )
    # Compute values
    perc_values = df.apply(
        lambda row: ram_usage_to_percentage(row[ram_col], row[node_col]),
        axis=1
    )

    # Find the index of ram_usage column
    insert_at = df.columns.get_loc("ram_usage_node_percentage") + 1

    # Insert new column at that position
    df.insert(insert_at, "ram_usage_node_perc_theor", perc_values)

    return df

In [ ]:
# Function to remove outliers
def remove_outliers(df):
  # Iterate over each target column and handle outliers
  functions_column = [col for col in df if col.startswith('rate')]
  targets = [col for col in df if (col.startswith('power_usage_') or col.startswith('cpu_usage_') or col.startswith('ram_usage_') or col.startswith('overloaded_node') or col.startswith('medium_latency')) and 'idle' not in col]
  grouped = df.groupby(functions_column + ['node_type'])
  threshold = 1
  for target in targets:
      print(target)
      if target != 'overloaded_node':
          mean = grouped[target].transform('mean')
          std = grouped[target].transform('std')
          outliers = (df[target] > mean + threshold * std) | (df[target] < mean - threshold * std)
          print(outliers.sum())
          df[target] = df[target].where(~outliers, mean)
      else:
          # Replace the overloaded value of the group by the mode.
          new_overloaded = grouped[target].transform(lambda x: x.mode().iloc[0])
          df['overloaded_node'] = new_overloaded
          print(df["overloaded_node"].value_counts())
  df_only_useful = df[functions_column + targets]
  return df

### Plotting and Printing

In [ ]:


def plot_overloaded_node_distribution_by_type(df):
    """
    Displays a grouped bar chart showing the count of overloaded (1) and non-overloaded (0) nodes
    for each node_type category, with value labels on top of bars.
    """
    # Group by node_type and overloaded_node, then count
    counts = df.groupby(['node_type', 'overloaded_node']).size().reset_index(name='count')

    # Convert overloaded_node to readable labels
    counts['overloaded_node'] = counts['overloaded_node'].map({0: 'Not Overloaded', 1: 'Overloaded'})

    # Define custom colors
    custom_palette = {
        'Not Overloaded': '#29af7f',   # viridis green
        'Overloaded': '#3b528b'       # viridis blue
    }

    # Plot
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(
        data=counts,
        x='node_type',
        y='count',
        hue='overloaded_node',
        palette=custom_palette
    )

    # Add bar labels (numbers on top)
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', label_type='edge', padding=3, fontsize=10)

    plt.title('Distribution of Overloaded Nodes by Node Type')
    plt.xlabel('Node Type')
    plt.ylabel('Number of Nodes')

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(title='Overload Status')
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_training_histories(histories, metric_map=None):
    """
    Plot training histories for multiple models, automatically handling different metrics.

    Parameters
    ----------
    histories : dict
        Dictionary where keys are model names (str) and values are Keras History objects.
        Example:
            {
                "JanossyRNN": history_regression,
                "DeepClassifier": history_classification
            }

    metric_map : dict, optional
        Mapping from model name to metric of interest.
        If None, the function tries to infer automatically.
        Example:
            {
                "JanossyRNN": "r2_score",
                "DeepClassifier": "accuracy"
            }
    """

    plt.figure(figsize=(9,6))

    for model_name, history in histories.items():
        hist = history.history

        # Try to detect metric automatically if not provided
        if metric_map is None or model_name not in metric_map:
            if 'r2_score' in hist:
                metric = 'r2_score'
            elif 'accuracy' in hist:
                metric = 'accuracy'
            elif 'mae' in hist:
                metric = 'mae'
            else:
                print(f"Could not find a known metric for {model_name}, skipping.")
                continue
        else:
            metric = metric_map[model_name]

        train_metric = hist.get(metric)
        val_metric = hist.get(f'val_{metric}')

        if train_metric is not None:
            plt.plot(train_metric, label=f'{model_name} - Train {metric}')
        if val_metric is not None:
            plt.plot(val_metric, linestyle='--', label=f'{model_name} - Val {metric}')

    plt.xlabel('Epochs')
    plt.ylabel('Metric Value')
    plt.title('Model Metrics over Epochs')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


In [ ]:


def plot_training_histories_separate(
    histories,
    metric_map=None,
    save_dir="figures_png/training_histories"
):
    """
    Plot each model's training metrics in separate figures and save them as PNG.
    """

    os.makedirs(save_dir, exist_ok=True)

    for model_name, history in histories.items():
        hist = history.history
        all_metrics = list(hist.keys())

        # Detect metrics if not provided
        if metric_map is None or model_name not in metric_map:
            metrics = sorted({m.replace('val_', '') for m in all_metrics if not m.startswith('val_')})
        else:
            metrics = metric_map[model_name]

        if not metrics:
            print(f"No metrics found for {model_name}, skipping.")
            continue

        # Plot each metric in a separate figure
        for metric in metrics:
            train_metric = hist.get(metric)
            val_metric = hist.get(f'val_{metric}')

            if train_metric is None and val_metric is None:
                print(f"Metric '{metric}' not found in {model_name}, skipping.")
                continue

            plt.figure(figsize=(8, 5))

            if train_metric is not None:
                plt.plot(train_metric, label='Train', linewidth=2)
            if val_metric is not None:
                plt.plot(val_metric, linestyle='--', label='Validation', linewidth=2)

            plt.xlabel('Epochs')
            plt.ylabel(metric)
            plt.title(f'{model_name} - {metric} over Epochs')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()

            # ---- SAVE FIGURE (PNG) ----
            filename = f"{model_name}_{metric}.png"
            filepath = os.path.join(save_dir, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')

            plt.show()
            plt.close()


In [ ]:
def add_row_to_df(df, row_dict):
    """
    Adds a single row to a DataFrame using a dictionary.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to which the row will be added.
    row_dict : dict
        Keys must match the DataFrame's columns.

    Returns
    -------
    pandas.DataFrame
        Updated DataFrame with the new row added.
    """
    return pd.concat([df, pd.DataFrame([row_dict])], ignore_index=True)

In [ ]:
def display_results(results_df):
  for task_name, df in results_df.items():
    printmd(f"# {task_name}")
    printmd("---")
    display(HTML(df.to_html()))

In [ ]:
def printmd(string):
    display(Markdown(string))

In [ ]:
# Function to obtain a barchart of the type of nodes
def plot_node_type_distribution(df):
    """
      Displays the number of data for each node type in a DataFrame using a bar chart.

      :param df: DataFrame containing the 'node_type' column with the node types.
    """
    # Count of occurrences for each node type
    node_counts = df['node_type'].value_counts().sort_index()

    # Bar chart

    fig, axes = plt.subplots(1, 1, figsize=(10, 6))
    node_counts.plot(kind='bar',ax=axes)
    plt.title('Distribution of Number of Data by Node Type')
    plt.xlabel('Node Type (0 = Heavy, 1 = Mid, 2 = Light)')
    plt.ylabel('Number of Rows')
    plt.xticks(rotation=0)  # Maintains names of horizontal node types
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    for i in axes.containers:
        axes.bar_label(i)

    plt.show()

In [ ]:
# Function to obtain a barchart of overloaded node distribution
def plot_overloaded_node_distribution(df):
    """
    Displays the number of rows where 'overloaded_node' is 1 and the number where it is 0.

    :param df: DataFrame containing column 'overloaded_node'.
    """
    # Count of occurrences of 0 and 1 in the 'overloaded_node' column
    overloaded_counts = df['overloaded_node'].value_counts()

    # Bar chart
    fig, axes = plt.subplots(1, 1, figsize=(8, 5))
    overloaded_counts.plot(kind='bar',ax=axes)
    plt.title('Distribution of Overloaded Nodes')
    plt.xlabel('Overload Status (0 = No, 1 = Yes)')
    plt.ylabel('Number of Lines')
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    for i in axes.containers:
      axes.bar_label(i)

    plt.show()

In [ ]:
# Function used to calculate metrics based on the task
def metrics(task_type, y_test, y_pred):
  if(task_type == 'regression'):
    mse = mean_squared_error(y_test, y_pred)
    print("mse:", mse)
    rmse = math.sqrt(mse)
    print("rmse:", rmse)
    mape = mean_absolute_percentage_error(y_test, y_pred)*100
    print("mape:", mape)
    r2 = r2_score(y_test, y_pred)
    print("R-squared score:", r2)
    std_dev = np.std(y_pred)
    print("Standard deviation:", std_dev)
    return mse, rmse, mape, r2, std_dev
  elif(task_type.endswith('classification')):
    y_pred = (y_pred > 0.5).astype(int)
    report = classification_report(y_test, y_pred)
    print("Classification Report:\n", report)
    return classification_report(y_test, y_pred, output_dict=True)

In [ ]:
def plot_regression(y_test, y_pred, target_name, save_dir="figures_png"):
    # Create output directory if it does not exist
    os.makedirs(save_dir, exist_ok=True)

    # ---------- Regression plot ----------

    try:
        m, q = np.polyfit(y_test.ravel(), y_pred.ravel(), 1)
    except Exception:
        if "power" in target_name:
            print(f"Expected exception for {target_name}")
        else:
            print("It was not possible to calculate the regression lines.")
        plt.close()
        return

    plt.plot(y_test, y_pred, 'o', color='red', fillstyle='none', label='Predictions')
    plt.plot(y_test, m * y_test + q, linestyle='--', label='Regression line')

    # Bisector (ideal behavior)
    xmin, xmax = 1, 2
    x_line = np.linspace(xmin, xmax, 100)
    plt.plot(x_line, x_line, '--', color='black', alpha=0.7, label='Ideal (y=x)')

    plt.xlabel('Observed values')
    plt.ylabel('Predicted values')
    plt.title(f'Regression of {target_name}')
    plt.xlim(1, 2)
    plt.ylim(1, 2)

    # Save regression plot (PNG)
    reg_path = os.path.join(save_dir, f"regression_{target_name}.png")
    plt.savefig(reg_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    # ---------- Residuals plot ----------

    residuals = y_test.flatten() - y_pred.flatten()

    sns.scatterplot(x=y_test.flatten(), y=residuals, label='Observations')
    plt.axhline(y=0, color='black', linestyle='--', linewidth=1)

    plt.xlabel('Observed values')
    plt.ylabel('Residuals (Observed − Predicted)')
    plt.title(f'Residuals - {target_name}')
    plt.legend()

    # Save residuals plot (PNG)
    res_path = os.path.join(save_dir, f"residuals_{target_name}.png")
    plt.savefig(res_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


In [ ]:
def show_result(all_predictions,y_test_dict,tasks,base_path,plot_path):
  results_df = {}
  for task, pred in all_predictions.items():
    for name, y_pred in pred.items():
      task_name = f"{task}_{name}"

      print(f"Task: {task_name}")

      os.makedirs(base_path, exist_ok=True)
      path = os.path.join(base_path, f"{task_name}.xlsx")
      df = pd.DataFrame()


      if "classification" in task_name:
          task_type = 'binary classification'
          # Convert DataFrame to NumPy array before slicing
          y_test = y_test_dict[task].values[:, -1]
      else:
          task_type = 'regression'
          # Convert DataFrame to NumPy array before slicing
          y_test = y_test_dict[task].values[:, :-1]


      y_test_df = pd.DataFrame(y_test, columns=tasks[task_name]["targets"])
      y_pred_df = pd.DataFrame(y_pred, columns=tasks[task_name]["targets"])
      # Overall evaluation
      # metrics(task_type, y_test_df, y_pred_df)
      if task_type == 'regression':
        mse, rmse, mape, r2, std_dev = metrics(task_type, y_test_df, y_pred_df)
        df = add_row_to_df(df, {'Target': "overall_model_performance",'MSE': mse,
                                                'RMSE': rmse, 'MAPE' : mape, 'R2': r2, 'Std Dev': ""})
      else:
        report = metrics(task_type, y_test_df, y_pred_df)
        df = pd.DataFrame(report).transpose()

        # Optional: Reset index for nicer formatting
        df = df.reset_index().rename(columns={'index': 'class'})


      # Individual Evaluation per Target

      if y_test_df.shape[1] > 1:
        for column in y_test_df.columns:
          print("-" * 30)
          print(f"Column: {column}")
          mse, rmse, mape ,r2, std_dev =  metrics(task_type, y_test_df[column], y_pred_df[column])
          plot_regression(y_test_df[column].values, y_pred_df[column].values, column,plot_path)
          df = add_row_to_df(df, {'Target': column,'MSE': mse,
                                                'RMSE': rmse, 'MAPE' : mape, 'R2': r2, 'Std Dev': std_dev})

      df.to_excel(path)
      results_df[task_name] = df



  print("-" * 30)
  return results_df

### Preprocessing for Model

In [ ]:
def remove_augmented_rows_from_test(x_test, y_test, augmented_df, features, verbose=True):
    """
    Remove rows from (x_test, y_test) where feature combinations already exist in augmented_df.

    Parameters
    ----------
    x_test : pd.DataFrame
        Test feature set.
    y_test : pd.DataFrame or pd.Series
        Test target set.
    augmented_df : pd.DataFrame
        DataFrame containing augmented samples (with same feature columns).
    features : list
        List of feature column names to use for matching.
    verbose : bool, default=True
        Whether to print debug information and stats.

    Returns
    -------
    x_test_filtered : pd.DataFrame
        Test features after removing overlapping rows.
    y_test_filtered : pd.DataFrame or pd.Series
        Corresponding test targets after filtering.
    """

    # --- Safety checks ---
    missing_in_test = [f for f in features if f not in x_test.columns]
    missing_in_aug = [f for f in features if f not in augmented_df.columns]
    if missing_in_test or missing_in_aug:
        raise ValueError(f"Missing columns — In test: {missing_in_test}, In augmented: {missing_in_aug}")

    if verbose:
        print(f"\nMatching test samples against augmented_df on {len(features)} features: {features}")

    # --- Drop duplicates from augmented features to speed up merge ---
    aug_unique = augmented_df[features].drop_duplicates()
    if verbose:
        print(f"Unique combinations in augmented_df: {len(aug_unique)} (from {len(augmented_df)})")

    # --- Identify overlapping rows ---
    merged = x_test.merge(aug_unique, on=features, how='left', indicator=True)

    merged.index = x_test.index  # re-align indices to match x_test

    overlaps_mask = merged['_merge'] == 'both'
    removed_mask = merged['_merge'] == 'left_only'

    n_total = len(x_test)
    n_overlaps = overlaps_mask.sum()
    n_remaining = removed_mask.sum()


    # --- Filter test set ---
    x_test_filtered = x_test[removed_mask].reset_index(drop=True)

    y_test_filtered = y_test.loc[removed_mask.values].reset_index(drop=True)

    unique_combos = x_test_filtered[features].drop_duplicates().shape[0]

    # --- Debug stats ---
    if verbose:
        print("\nTest Filter Summary:")
        print(f"   Total test samples      : {n_total}")
        print(f"   Overlaps with augmented  : {n_overlaps}")
        print(f"   Remaining test samples   : {n_remaining}")
        print(f"   Rows removed percentage  : {n_overlaps / n_total * 100:.2f}%")
        print(f"   Unique feature combinations    : {unique_combos}")

    return x_test_filtered, y_test_filtered


In [ ]:


def split_test_nodewise(X_test_dict, Y_test_dict, tasks):
    """
    Split test data into nodewise subsets for both regression and classification tasks.

    Handles 3D regression input (seq_len, feature_dim) and 2D classification input.

    Returns
    -------
    X_test_nodewise : dict
        {'regression': {node_type: X_subset, ...}, 'classification': {...}}
    Y_test_nodewise : dict
        {'regression': {node_type: y_subset, ...}, 'classification': {...}}
    """
    node_types = [0.0, 1.0, 2.0]
    X_test_nodewise = {}
    Y_test_nodewise = {}

    for node in node_types:
      X_test_nodewise[node] = {}
      Y_test_nodewise[node] = {}

    for task_name in tasks:

      if task_name.startswith('overloaded'):
        # --- Classification (2D input) ---
        X_cls, y_cls = X_test_dict[task_name], Y_test_dict[task_name]
        # node_type = first element of each row
        node_col_cls = X_cls[:, 0]
        for node in node_types:
            mask = node_col_cls == node
            X_test_nodewise[node][task_name] = X_cls[mask]
            Y_test_nodewise[node][task_name] = y_cls[mask]

      else:
        # --- Regression (3D input) ---
        X_reg, y_reg = X_test_dict[task_name], Y_test_dict[task_name]
        # node_type = last element of last time step for each sample
        node_col_reg = np.array([x[-1, -1] for x in X_reg])
        for node in node_types:
            mask = node_col_reg == node
            X_test_nodewise[node][task_name] = X_reg[mask]
            Y_test_nodewise[node][task_name] = y_reg[mask]


    return X_test_nodewise, Y_test_nodewise


In [ ]:

def check_node_type_distribution(X_train, X_val, X_test, node_col='node_type', normalize=True):
    """
    Check and compare the distribution of `node_type` (or any categorical feature)
    across train, validation, and test splits.

    Parameters
    ----------
    X_train, X_val, X_test : pd.DataFrame
        DataFrames for training, validation, and test sets.
    node_col : str, default='node_type'
        Name of the column representing node types (or any categorical variable).
    normalize : bool, default=True
        If True, show proportions (%) instead of raw counts.

    Returns
    -------
    pd.DataFrame
        Summary table comparing distributions across splits.
    """
    # Validate column existence
    for split_name, df in zip(['Train', 'Val', 'Test'], [X_train, X_val, X_test]):
        if node_col not in df.columns:
            raise ValueError(f"'{node_col}' not found in {split_name} set.")

    # Compute distributions
    def get_dist(df, name):
        return df[node_col].value_counts(normalize=normalize).rename(name)

    dist_train = get_dist(X_train, 'Train')
    dist_val   = get_dist(X_val, 'Val')
    dist_test  = get_dist(X_test, 'Test')

    # Combine into one comparison DataFrame
    dist_summary = pd.concat([dist_train, dist_val, dist_test], axis=1).fillna(0)
    if normalize:
        dist_summary = dist_summary * 100  # show as %

    # Add total rows for reference
    total_counts = {
        "Train Total": len(X_train),
        "Val Total": len(X_val),
        "Test Total": len(X_test)
    }
    totals_df = pd.DataFrame(total_counts, index=["Total Rows"])

    printmd("## Node Type Distribution Across Splits:")
    print(dist_summary.round(2))
    print("\n Dataset Sizes:")
    print(totals_df)

    return dist_summary


In [ ]:
def check_data_leakage(X_train, X_val, X_test, group_cols=None):
    """
    Verify that there is no data leakage (i.e., no repeated feature combinations)
    across train, validation, and test splits.

    Parameters
    ----------
    X_train, X_val, X_test : pd.DataFrame
        DataFrames for train, validation, and test sets.
    group_cols : list of str, optional
        Columns to consider for checking duplicates.
        If None, all columns of X_train are used.

    Returns
    -------
    None
        Prints leakage report. Raises AssertionError if leakage is detected.
    """


    if group_cols is None:
        group_cols = X_train.columns.tolist()

    # Create a unique signature for each feature combination
    def make_signatures(df):
        return set(df[group_cols].astype(str).agg('-'.join, axis=1))

    sig_train = make_signatures(X_train)
    sig_val   = make_signatures(X_val)
    sig_test  = make_signatures(X_test)

    # Check overlaps
    leak_train_val = sig_train.intersection(sig_val)
    leak_train_test = sig_train.intersection(sig_test)
    leak_val_test = sig_val.intersection(sig_test)

    total_leaks = len(leak_train_val) + len(leak_train_test) + len(leak_val_test)

    printmd("## Data Leakage Check:")

    print(f"Train Samples: {len(X_train)}")
    print(f"Val Samples: {len(X_val)}")
    print(f"Test Samples: {len(X_test)}")

    print("\n")
    # print(f"- Train–Val overlaps: {len(leak_train_val)}")
    # print(f"- Train–Test overlaps: {len(leak_train_test)}")
    # print(f"- Val–Test overlaps: {len(leak_val_test)}")

    if total_leaks == 0:
        print("No data leakage detected. Splits are clean.")
    else:
        print("Data leakage detected!")
        if len(leak_train_val):
            print(f"  → {len(leak_train_val)} overlapping feature groups between Train and Val")
        if len(leak_train_test):
            print(f"  → {len(leak_train_test)} overlapping feature groups between Train and Test")
        if len(leak_val_test):
            print(f"  → {len(leak_val_test)} overlapping feature groups between Val and Test")
        #raise AssertionError("Data leakage detected between splits.")


In [ ]:
def limit_identical_combinations(df, features, target_cols, max_per_combination=1):
    """
    Limit the dataset to at most `max_per_combination` identical feature combinations.

    Parameters
    ----------
    df : pd.DataFrame
    features : list
        Columns to group by when identifying identical combinations.
    target_cols : list
        Target column names.
    max_per_combination : int
        Maximum allowed identical combinations per group.

    Returns
    -------
    df_limited : pd.DataFrame
        Reduced datasets after limiting duplicates.
    """
    print(f"\n=== Limiting to max {max_per_combination} identical combinations ===")

    # df_combined = pd.concat([X, y], axis=1)
    df_limited = (
        df.groupby(features, group_keys=False)
        .apply(lambda x: x.head(max_per_combination))
        .reset_index(drop=True)
    )

    X_limited = df_limited[features]
    y_limited = df_limited[target_cols]

    print(f"Original row count: {len(df)}")
    print(f"After limiting: {len(df_limited)} (removed {len(df) - len(df_limited)})")
    print("\nClass distribution after limiting:")
    print(y_limited.value_counts())

    return df_limited

In [ ]:
def generate_synthetic_overloaded(df, features, target_col):
    """
    Generate synthetic overloaded samples to balance classes per node_type.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataset (after limiting).
    features : list
        Feature column names.
    target_col : str
        Binary target column name (0 = non-overloaded, 1 = overloaded).

    Returns
    -------
    df_augmented : pd.DataFrame
        Dataset containing newly generated synthetic rows.
    """
    print("\n=== Generating synthetic overloaded samples (Unique Features Balancing) ===")

    rate_cols = [col for col in df.columns if col.startswith("rate_function_")]
    node_types = df["node_type"].unique()
    augmented_rows = []
    generated_total = 0

    existing_keys = set(tuple(row[col] for col in features) for _, row in df.iterrows())

    for nt in node_types:
        subset = df[df["node_type"] == nt]
        count_1 = subset[subset[target_col] == 1].shape[0]
        count_0 = subset[subset[target_col] == 0].shape[0]
        needed = count_0 - count_1

        print(f"\n[Node Type {nt}] Overloaded: {count_1}, Non-overloaded: {count_0}")
        if needed <= 0:
            print(f"Already balanced or overloaded.")
            continue

        print(f"Generating {needed} synthetic samples for node_type {nt}")
        df_overloaded = subset[subset[target_col] == 1]
        generated = 0

        for _, row in df_overloaded.iterrows():
            overloaded_funcs = [col for col in rate_cols if row[col] > 0]
            if len(overloaded_funcs) != 3:
                continue

            fixed_rates = {col: row[col] for col in overloaded_funcs}

            for target_func in overloaded_funcs:
                start = int(fixed_rates[target_func])
                for val in range(start + 2, 201, 5):
                    if generated >= needed:
                        break

                    new_row = {col: 0.0 for col in rate_cols}
                    for col in overloaded_funcs:
                        new_row[col] = val if col == target_func else fixed_rates[col]

                    # Fill metadata and targets
                    new_row["node_type"] = nt
                    new_row[target_col] = 1

                    # Add other columns from original
                    for col in df.columns:
                        if col not in new_row and col not in rate_cols:
                            new_row[col] = row[col]

                    new_key = tuple(new_row[col] for col in features)
                    if new_key not in existing_keys:
                        augmented_rows.append(new_row)
                        existing_keys.add(new_key)
                        generated += 1
                        generated_total += 1

                if generated >= needed:
                    break
            if generated >= needed:
                break

        print(f"Generated {generated} samples for node_type {nt}")

    print(f"\n Total synthetic samples generated: {generated_total}")
    return pd.DataFrame(augmented_rows)

In [ ]:
def balance_df_nodewise(df, df_augmented, node_col='node_type', target_col='overloaded_node'):
    """
    Balance df within each node_type based on overloaded_node (target).
    df_augmented contains only rows with overloaded_node = 1.
    Includes detailed debug prints.
    """

    balanced_parts = []
    node_types = df[node_col].unique()

    print("\n========= ROWS BALANCING =========\n")
    print(f"Node types found: {list(node_types)}\n")

    for nt in node_types:
        print(f"\n--- Processing node_type: {nt} ---")

        subset = df[df[node_col] == nt].copy()

        # Count original samples
        count_1 = (subset[target_col] == 1).sum()
        count_0 = (subset[target_col] == 0).sum()

        print(f"Original counts → overloaded=0: {count_0}, overloaded=1: {count_1}")

        # Check if balance is needed
        if count_1 >= count_0:
            print("Already balanced or overloaded=1 is majority. No augmentation needed.")
            balanced_parts.append(subset)
            continue

        needed = count_0 - count_1
        print(f"Need to add {needed} more rows for overloaded=1")

        # Filter augmented rows for this node_type
        aug_rows = df_augmented[df_augmented[node_col] == nt]

        if aug_rows.empty:
            print(f"[WARNING] df_augmented has NO rows for node_type={nt}. Cannot balance this group!")
            balanced_parts.append(subset)
            continue

        print(f"Augmented rows available for this node_type={nt}: {len(aug_rows)}")

        # Calculate repeat factor
        repeats = ceil(needed / len(aug_rows))
        print(f"Repeating augmented rows {repeats} times")

        repeated = pd.concat([aug_rows] * repeats, ignore_index=True)

        rows_to_add = repeated.iloc[:needed]
        print(f"Rows actually added: {len(rows_to_add)}")

        # Append augmented rows
        subset = pd.concat([subset, rows_to_add], ignore_index=True)

        # Verify new counts
        new_count_1 = (subset[target_col] == 1).sum()
        new_count_0 = (subset[target_col] == 0).sum()
        print(f"After augmentation → overloaded=0: {new_count_0}, overloaded=1: {new_count_1}")

        balanced_parts.append(subset)

    print("\n========= BALANCING COMPLETED =========\n")
    final_df = pd.concat(balanced_parts, ignore_index=True)
    return final_df


In [ ]:
def perform_custom_oversampling(df, features, reg_target, class_target):
    """
    Custom domain-aware oversampling replacing SMOTENC.

    Steps:
    1. Limit identical feature combinations.
    2. Generate synthetic overloaded samples per node_type to balance unique features(Group level balance).
    3. Row-level Balance by duplicating augmented rows and returning a balanced dataset.

    Parameters
    ----------
    df : pd.DataFrame
    features : list
        Feature columns used for grouping and synthesis.
    reg_target : list
        Regression Target column(s).
    class_target : list
        Classification Target column(s).

    Returns
    -------
    X_resampled, y_resampled : pd.DataFrame, pd.DataFrame
        Balanced feature and target datasets.
    """
    # Step 1: Apply limiting rule
    df_class = df[features + class_target]
    df_limited = limit_identical_combinations(df_class, features, class_target)

    # Step 2: Combine for rebalancing
    # df_limited = pd.concat([X_limited, y_limited], axis=1)
    target_col = class_target[0]

    # Step 3: Generate new samples to balance limited dataset
    df_augmented = generate_synthetic_overloaded(df_limited, features, target_col)

    # Optionally save the augmented rows
    os.makedirs(PATH_TO_AUGMENTED_ROWS, exist_ok=True)
    df_augmented.to_csv(PATH_TO_AUGMENTED_ROWS + "augmented_overloaded_samples.csv", index=False)
    print("Augmented rows saved to 'augmented_overloaded_samples.csv'")

    # Step 4: Final overall balance dataset (repeated features)
    df_balanced = balance_df_nodewise(df, df_augmented)
    X_res = df_balanced[features]
    y_res = df_balanced[reg_target + class_target]

    print("\nFinal class distribution after rebalancing:")
    print(y_res[class_target].value_counts())
    print("\nFinal class distribution after rebalancing (node_type-wise):")
    print(df_balanced.groupby("node_type")["overloaded_node"].value_counts().unstack(fill_value=0))

    print("\nFinal node_type distribution:")
    print(X_res["node_type"].value_counts())

    print("\n========= FINAL UNIQUE FEATURES SUMMARY =========")

    # Convert to tuples for unique combinations
    all_unique = df_balanced[features].drop_duplicates()
    unique_0 = df_balanced[df_balanced[target_col] == 0][features].drop_duplicates()
    unique_1 = df_balanced[df_balanced[target_col] == 1][features].drop_duplicates()

    print(f"Total unique feature combinations       : {len(all_unique)}")
    print(f"Unique feature combinations for 0       : {len(unique_0)}")
    print(f"Unique feature combinations for 1       : {len(unique_1)}")

    print("==========================================\n")

    return X_res, y_res, df_augmented, df_balanced

In [ ]:
def perform_oversampling(X, y, categorical_features, target_name=None, random_state=42):
    """
    Apply SMOTENC oversampling to handle class imbalance in datasets
    containing categorical features. Prints before/after class distributions.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature set.
    y : pandas.Series
        Target variable.
    categorical_features : list of str
        Names of categorical columns in X.
    target_name : str, optional
        Name of the target for display/logging.
    random_state : int, default=42
        Random seed for reproducibility.

    Returns
    -------
    X_resampled, y_resampled : pandas.DataFrame, pandas.Series
        Oversampled feature and target datasets.
    """
    if target_name:
        print(f"\nPerforming SMOTENC for target: {target_name}")
    print("Class distribution before SMOTE:")
    print(y.value_counts())

    cat_indices = [X.columns.get_loc(col) for col in categorical_features]
    smote = SMOTENC(categorical_features=cat_indices, random_state=random_state)

    try:
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("Class distribution after SMOTE:")
        print(y_resampled.value_counts())
        print("Categorical feature distribution after SMOTE:")
        for col in categorical_features:
            print(f"\n{col}:\n{X_resampled[col].value_counts()}")
    except Exception as e:
        print(f"Could not perform SMOTE: {e}")
        return X, y

    return X_resampled, y_resampled


In [ ]:
def prepare_feature_target_datasets(df, tasks):
    """
    Prepare datasets for multiple targets regression and overloaded node classification
    with optional SMOTENC oversampling.

    Parameters
    ----------
    df : pandas.DataFrame
        Full dataset containing both features and targets.
    features : list of str
        List of feature column names.
    targets : list of str
        List of target column names.
    categorical_features : list of str
        Categorical feature columns to be passed to SMOTENC.

    Returns
    -------
    features_datasets : dict
        Mapping target names to feature DataFrames (oversampled if needed).
    target_datasets : dict
        Mapping target names to target Series.
    """

    feature_dataset = {}
    target_dataset = {}
    df_augmented = None
    for task_name, task_info in tasks.items():
      # X = df[task_info["features"]]
      # y = df[task_info["targets"]]
      if "Multi_Task" in task_name:
        printmd(f"##{task_name}")
        # X, y = perform_oversampling( X, y, categorical_features, target_name=task_name)
        X, y, df_augmented, df_balanced = perform_custom_oversampling(df, task_info["features"], task_info["regression_targets"],
                                                         task_info["classification_targets"])
      feature_dataset[task_name] = X
      target_dataset[task_name] = y


    return feature_dataset, target_dataset, df_augmented, df_balanced


In [ ]:
def split_train_test(X, y, test_size=0.2, random_state=42, stratify=None):
    """
    Perform a simple train–test split.

    Parameters
    ----------
    X : pandas.DataFrame or numpy.ndarray
        Feature set.
    y : pandas.Series or numpy.ndarray
        Target variable.
    test_size : float, default=0.2
        Proportion of dataset to include in the test split.
    random_state : int, default=42
        Random seed for reproducibility.
    stratify : array-like, default=None
        If not None, data is split in a stratified fashion (useful for classification).

    Returns
    -------
    X_train, X_test, y_train, y_test : tuple
        Split datasets for training and testing.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )
    return X_train, X_test, y_train, y_test

In [ ]:
def split_train_val_test(X, y, val_size=0.1, test_size=0.2, random_state=42, stratify=None):
    """
    Perform a train–validation–test split.

    Parameters
    ----------
    X : pandas.DataFrame or numpy.ndarray
        Feature set.
    y : pandas.Series or numpy.ndarray
        Target variable.
    val_size : float, default=0.1
        Proportion of dataset for validation (from the remaining after test split).
    test_size : float, default=0.2
        Proportion of dataset for testing.
    random_state : int, default=42
        Random seed for reproducibility.
    stratify : array-like, default=None
        If not None, data is split in a stratified fashion (useful for classification).

    Returns
    -------
    X_train, X_val, X_test, y_train, y_val, y_test : tuple
        Split datasets for training, validation, and testing.
    """
    # First, split off the test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )

    # Adjust validation size relative to remaining data
    val_relative_size = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_relative_size,
        random_state=random_state,
        stratify=y_temp if stratify is not None else None
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
def split_train_val_test_grouped(X, y, val_size=0.1, test_size=0.2,
                                 random_state=42, stratify=None, group_cols=None):
    """
    Perform a train–validation–test split while ensuring no feature-level data leakage
    (i.e., identical feature rows stay in the same split).

    Parameters
    ----------
    X : pd.DataFrame
        Feature matrix.
    y : pd.Series or pd.DataFrame
        Target variable(s).
    val_size : float, default=0.1
        Validation set proportion (relative to remaining after test split).
    test_size : float, default=0.2
        Test set proportion of full dataset.
    random_state : int, default=42
        Random seed for reproducibility.
    stratify : array-like, default=None
        Optional stratification column/array.
    group_cols : list of str, optional
        Columns to define "uniqueness" of feature combinations.
        If None, all columns of X are used.

    Returns
    -------
    X_train, X_val, X_test, y_train, y_val, y_test : tuple
        Group-consistent splits (no duplicate feature combinations across sets).
    """
    if not isinstance(X, pd.DataFrame):
        raise ValueError("X must be a pandas DataFrame for group-based split.")

    if group_cols is None:
        group_cols = X.columns.tolist()

    # Step 1: Identify unique feature combinations
    X_unique = X[group_cols].drop_duplicates().reset_index(drop=True)

    # Step 2: Map each unique combination to its group index
    X['__group_id__'] = X[group_cols].astype(str).agg('-'.join, axis=1)
    group_ids = X_unique.astype(str).agg('-'.join, axis=1)

    # Step 3: Split group IDs (ensuring no overlap)
    group_train, group_test = train_test_split(
        group_ids,
        test_size=test_size,
        random_state=random_state,
        stratify=None  # stratify not possible directly on groups
    )

    # Adjust validation size relative to train
    val_relative_size = val_size / (1 - test_size)
    group_train, group_val = train_test_split(
        group_train,
        test_size=val_relative_size,
        random_state=random_state,
        stratify=None
    )

    # Step 4: Assign each sample based on group membership
    def mask_from_groups(groups):
        return X['__group_id__'].isin(groups)

    train_mask = mask_from_groups(group_train)
    val_mask = mask_from_groups(group_val)
    test_mask = mask_from_groups(group_test)

    # Step 5: Create splits
    X_train, X_val, X_test = X[train_mask].drop(columns='__group_id__'), X[val_mask].drop(columns='__group_id__'), X[test_mask].drop(columns='__group_id__')
    y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

    # Final checks
    print(f"\nSplit Summary (group-consistent):")
    print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")
    print(f"Unique feature groups: {len(group_ids)} (train={len(group_train)}, val={len(group_val)}, test={len(group_test)})")

    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
# def fit_minmax_scaler(X, y=None):
#     """
#     Fit MinMaxScaler for features (and optionally targets).

#     Parameters
#     ----------
#     X : pd.DataFrame
#         Feature dataset.
#     y : pd.Series or pd.DataFrame, optional
#         Target dataset (optional).

#     Returns
#     -------
#     scaler_x : MinMaxScaler
#         Fitted scaler for features.
#     scaler_y : MinMaxScaler or None
#         Fitted scaler for targets, or None if y not provided.
#     """
#     scaler_x = MinMaxScaler()
#     scaler_x.fit(X)

#     scaler_y = None
#     if y is not None:
#         scaler_y = MinMaxScaler()
#         scaler_y.fit(y)

#     print("[INFO] MinMaxScaler fitted:")
#     print(f"  - Features shape: {X.shape}")
#     if y is not None:
#         print(f"  - Targets shape: {y.shape}")
#     print()

#     return scaler_x, scaler_y

In [ ]:
def fit_minmax_scaler(X, y=None, regression_cols=None, classification_col=None):
    """
    Fit MinMaxScaler for features (and optionally regression targets).

    X:
      - node_type column is excluded from scaling

    y:
      - only regression columns are scaled
      - classification column remains unchanged
    """

    # ----------------------
    # 1. Handle feature scaler
    # ----------------------
    scaler_x = MinMaxScaler(feature_range=(1,2))
    # Drop node_type if exists
    if "node_type" in X.columns:
        scaler_x.fit(X.drop(columns=["node_type"]))
    else:
        scaler_x.fit(X)

    # ----------------------
    # 2. Handle target scaler
    # ----------------------
    scaler_y = None
    if y is not None and regression_cols is not None:
        # Fit scaler on regression columns only
        scaler_y = MinMaxScaler(feature_range=(1,2))
        scaler_y.fit(y[regression_cols])

    # ---- Logging ----
    print("[INFO] MinMaxScaler fitted:")
    print(f"  - Features shape: {X.shape}")
    if y is not None:
        print(f"  - Regression targets shape: {y[regression_cols].shape}")
        print(f"  - Classification column '{classification_col}' excluded from scaling")
    print()

    return scaler_x, scaler_y


In [ ]:
def save_scalers(scaler_x, scaler_y, task_name, base_path='./scalers/functions/'):
    """
    Save fitted scalers for features and targets.

    Parameters
    ----------
    scaler_x : MinMaxScaler
        Fitted feature scaler.
    scaler_y : MinMaxScaler or None
        Fitted target scaler (if any).
    task_name : str
        Name of the task (e.g., 'Multi_Target_regression', 'overloaded_node_classification')
    base_path : str, default='./scalers/functions/'
        Base directory to save scalers.
    """
    # Create directories
    x_dir = os.path.join(base_path, 'scaler_x')
    y_dir = os.path.join(base_path, 'scaler_y')
    os.makedirs(x_dir, exist_ok=True)
    os.makedirs(y_dir, exist_ok=True)

    # Save X scaler
    joblib.dump(scaler_x, os.path.join(x_dir, f"{task_name}_x.joblib"))

    # Save Y scaler (only if provided)
    if scaler_y is not None:
        joblib.dump(scaler_y, os.path.join(y_dir, f"{task_name}_y.joblib"))

    print(f"[INFO] Scalers saved for target '{task_name}'")
    if scaler_y is None:
        print("  - No target scaler (classification or categorical target).")
    print()


In [ ]:
# def transform_with_scalers(
#     X_train, X_val, X_test, scaler_x,
#     y_train=None, y_val=None, y_test=None, scaler_y=None
# ):
#     """
#     Transform train, validation, and test sets using fitted scalers.

#     Parameters
#     ----------
#     X_train, X_val, X_test : pd.DataFrame
#         Feature datasets to transform.
#     scaler_x : MinMaxScaler
#         Fitted scaler for features.
#     y_train, y_val, y_test : pd.Series or pd.DataFrame, optional
#         Target datasets to transform.
#     scaler_y : MinMaxScaler or None
#         Fitted scaler for targets.

#     Returns
#     -------
#     X_train_scaled, X_val_scaled, X_test_scaled
#     y_train_scaled, y_val_scaled, y_test_scaled
#     """
#     # Scale X
#     X_train_scaled = scaler_x.transform(X_train)
#     X_val_scaled = scaler_x.transform(X_val)
#     X_test_scaled = scaler_x.transform(X_test)

#     # Scale Y (if applicable)
#     if scaler_y is not None and y_train is not None:
#         y_train_scaled = scaler_y.transform(y_train)
#         y_val_scaled = scaler_y.transform(y_val)
#         y_test_scaled = scaler_y.transform(y_test)
#     else:
#         y_train_scaled, y_val_scaled, y_test_scaled = y_train, y_val, y_test

#     print("[INFO] Transformation complete.")
#     print(f"  - X_train shape: {X_train_scaled.shape}")
#     print(f"  - X_val shape: {X_val_scaled.shape}")
#     print(f"  - X_test shape: {X_test_scaled.shape}\n")

#     return (
#         X_train_scaled, X_val_scaled, X_test_scaled,
#         y_train_scaled, y_val_scaled, y_test_scaled
#     )


In [ ]:
def transform_with_scalers(
    X_train, X_val, X_test, scaler_x,
    y_train=None, y_val=None, y_test=None,
    scaler_y=None, regression_cols=None, classification_col=None
):
    """
    Transform train/val/test using fitted scalers.

    X:
      - scales all columns except node_type
      - adds node_type back after scaling

    y:
      - scales only regression targets
      - keeps classification target unchanged
    """

    # =========================================
    # 1. Separate node_type from X
    # =========================================
    def split_node_type(df):
        if "node_type" in df.columns:
            node = df["node_type"].values.reshape(-1, 1)
            df_wo = df.drop(columns=["node_type"])
        else:
            node = None
            df_wo = df
        return df_wo, node


    X_train_noNT, node_train = split_node_type(X_train)
    X_val_noNT, node_val = split_node_type(X_val)
    X_test_noNT, node_test = split_node_type(X_test)

    # =========================================
    # 2. Scale numeric features only
    # =========================================
    X_train_scaled = scaler_x.transform(X_train_noNT)
    X_val_scaled   = scaler_x.transform(X_val_noNT)
    X_test_scaled  = scaler_x.transform(X_test_noNT)

    # Add back node_type (not scaled)
    if node_train is not None:
        X_train_scaled = np.hstack([node_train, X_train_scaled])
        X_val_scaled   = np.hstack([node_val, X_val_scaled ])
        X_test_scaled  = np.hstack([node_test, X_test_scaled])

    # =========================================
    # 3. Handle Y scaling
    # =========================================
    if y_train is not None and scaler_y is not None:

        # Extract and scale regression columns
        y_train_reg_s = scaler_y.transform(y_train[regression_cols])
        y_val_reg_s   = scaler_y.transform(y_val[regression_cols])
        y_test_reg_s  = scaler_y.transform(y_test[regression_cols])

        # Recombine regression + classification
        y_train_scaled = pd.DataFrame(y_train_reg_s, columns=regression_cols)
        y_val_scaled   = pd.DataFrame(y_val_reg_s, columns=regression_cols)
        y_test_scaled  = pd.DataFrame(y_test_reg_s, columns=regression_cols)

        # Add classification target unchanged
        y_train_scaled[classification_col] = y_train[classification_col].values
        y_val_scaled[classification_col]   = y_val[classification_col].values
        y_test_scaled[classification_col]  = y_test[classification_col].values

    else:
        y_train_scaled, y_val_scaled, y_test_scaled = y_train, y_val, y_test

    # ---- Logging ----
    print("[INFO] Transformation complete.")
    print(f"  - X_train shape: {X_train_scaled.shape}")
    print(f"  - X_val shape: {X_val_scaled.shape}")
    print(f"  - X_test shape: {X_test_scaled.shape}\n")

    return (
        X_train_scaled, X_val_scaled, X_test_scaled,
        y_train_scaled, y_val_scaled, y_test_scaled
    )


In [ ]:
# def transform_with_scalers_crossdom(X, scaler_x, y=None, scaler_y=None):
#     """
#     Transform full feature and target datasets using fitted scalers
#     for cross-domain evaluation (no train/val/test split).

#     Parameters
#     ----------
#     X : pd.DataFrame
#         Full feature dataset to transform.
#     scaler_x : MinMaxScaler
#         Fitted scaler for features.
#     y : pd.Series or pd.DataFrame, optional
#         Full target dataset to transform.
#     scaler_y : MinMaxScaler or None, optional
#         Fitted scaler for targets.

#     Returns
#     -------
#     X_scaled : np.ndarray
#         Scaled feature dataset.
#     y_scaled : np.ndarray or None
#         Scaled target dataset (if applicable).
#     """
#     # Scale X
#     X_scaled = scaler_x.transform(X)

#     # Scale Y (if applicable)
#     if scaler_y is not None and y is not None:
#         y_scaled = scaler_y.transform(y)
#     else:
#         y_scaled = y

#     print("[INFO] Cross-domain transformation complete.")
#     print(f"  - X shape: {X_scaled.shape}")
#     if y is not None:
#         print(f"  - y shape: {np.array(y_scaled).shape}\n")

#     return X_scaled, y_scaled


In [ ]:
def transform_with_scalers_crossdom(
    X, scaler_x,
    y=None, scaler_y=None,
    regression_cols=None,
    classification_col=None
):
    """
    Transform full feature and target datasets using fitted scalers
    for cross-domain evaluation (no train/val/test split).

    X:
      - scales all columns except node_type
      - adds node_type back as first column

    y:
      - scales only regression targets
      - keeps classification target unchanged
    """

    # =========================================
    # 1. Separate node_type from X
    # =========================================
    def split_node_type(df):
        if "node_type" in df.columns:
            node = df["node_type"].values.reshape(-1, 1)
            df_wo = df.drop(columns=["node_type"])
        else:
            node = None
            df_wo = df
        return df_wo, node

    X_noNT, node_vals = split_node_type(X)


    # =========================================
    # 2. Scale numeric features only
    # =========================================
    X_scaled_core = scaler_x.transform(X_noNT)

    # Add back node_type at the front
    if node_vals is not None:
        X_scaled = np.hstack([node_vals, X_scaled_core])
    else:
        X_scaled = X_scaled_core

    # =========================================
    # 3. Handle target scaling
    # =========================================
    if y is not None and scaler_y is not None:

        # Extract regression and classification
        y_reg_s = scaler_y.transform(y[regression_cols])

        # Reconstruct DataFrame
        y_scaled_df = pd.DataFrame(y_reg_s, columns=regression_cols)

        # Add classification as-is
        y_scaled_df[classification_col] = y[classification_col].values

        y_scaled = y_scaled_df

    else:
        y_scaled = y

    # =========================================
    # 4. Logging
    # =========================================
    print("[INFO] Cross-domain scaling complete.")
    print(f"  - X_scaled shape: {X_scaled.shape}")
    if y is not None:
        print(f"  - y_scaled shape: {np.array(y_scaled).shape}")

    return X_scaled, y_scaled


In [ ]:
def transform(df, rate_columns=None, pad_value=0.0):
    """
    Transform the input dataframe into padded, variable-length sequences.

    Each sample becomes a sequence of steps where each step = [rate, method_onehot(6), node_type].
    Steps with zero rate are removed. Then sequences are padded to the same length.

    Returns:
        X_padded: np.ndarray, shape (num_samples, max_seq_len, 8)
        mask: np.ndarray, shape (num_samples, max_seq_len), 1 for valid, 0 for padded
    """

    if rate_columns is None:
        rate_columns = [
            'rate_function_env', 'rate_function_curl', 'rate_function_eat_memory',
            'rate_function_nmap', 'rate_function_shasum', 'rate_function_figlet'
        ]

    node_type = df['node_type'].values
    num_samples = df.shape[0]
    sequence_length = len(rate_columns)  # 6

    # One-hot encode method indices (0–5)
    method_ids = np.arange(sequence_length).reshape(-1, 1)
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
    method_onehot = encoder.fit_transform(method_ids)  # (6, 6)

    X_sequences = []

    for i in range(num_samples):
        node_val = node_type[i]
        rate_values = df.loc[i, rate_columns].values  # shape (6,)

        # Keep only non-zero steps
        nonzero_mask = rate_values != 1
        if not np.any(nonzero_mask):
            nonzero_mask = np.array([True])  # Keep one dummy step if all zero

        rate_seq = rate_values[nonzero_mask].reshape(-1, 1)
        method_seq = method_onehot[nonzero_mask]
        node_seq = np.full((np.sum(nonzero_mask), 1), node_val)

        seq = np.concatenate([rate_seq, method_seq, node_seq], axis=1)  # (L_i, 8)
        X_sequences.append(seq)

    # Pad all sequences to same length (post-padding with zeros)
    max_len = max(len(seq) for seq in X_sequences)
    X_padded = pad_sequences(X_sequences, maxlen=max_len, dtype='float32', padding='post', value=pad_value)

    # Create mask (1 for real steps, 0 for padded)
    mask = np.array([[1]*len(seq) + [0]*(max_len - len(seq)) for seq in X_sequences], dtype='float32')

    # Drop node_type column if you want to reuse df
    df = df.drop(columns=['node_type'])

    return X_padded


In [ ]:
def preprocessing(feature_dataset, target_dataset, tasks, base_path, augmented_df):
    """
    Perform full preprocessing pipeline:
    - Split data into train/validation/test sets
    - Fit MinMax scalers for features and (optionally) targets
    - Save scalers to disk
    - Transform datasets using fitted scalers
    - Return scaled datasets and scalers for all tasks
    """

    # -----------------------------------------------------
    # Initialize dictionaries to store splits and scalers
    # -----------------------------------------------------
    x_train_dict, x_val_dict, x_test_dict = {}, {}, {}
    y_train_dict, y_val_dict, y_test_dict = {}, {}, {}
    x_scalers, y_scalers = {}, {}

    # -----------------------------------------------------
    # Loop through each task for preprocessing
    # -----------------------------------------------------
    for task_name,task_info in tasks.items():
        printmd(f"# {task_name}")
        printmd("---")
        # Extract feature and target datasets for this task
        X = feature_dataset[task_name].copy()
        y = target_dataset[task_name].copy()

        # -------------------------------------------------
        # Split into train, validation, and test sets
        # Use stratified split only for classification tasks
        # -------------------------------------------------
        X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test_grouped(
            X, y,
            val_size=0.1,
            test_size=0.4,
            random_state=42,
            stratify=y if task_name == "overloaded_node_classification" else None
        )

        X_test, y_test = remove_augmented_rows_from_test(X_test, y_test, augmented_df, features)
        X_val, y_val = remove_augmented_rows_from_test(X_val, y_val, augmented_df, features)

        # -------------------------------------------------
        # Check for data leakage
        # -------------------------------------------------
        check_node_type_distribution(X_train,X_val,X_test)
        check_data_leakage(X_train,X_val,X_test)

        # -------------------------------------------------
        # Fit MinMaxScaler
        # Skip target scaling for classification tasks
        # -------------------------------------------------
        printmd("## Scalaing")
        if task_name == "overloaded_node_classification":
            scaler_x, scaler_y = fit_minmax_scaler(X_train)
        else:
            scaler_x, scaler_y = fit_minmax_scaler(X_train, y_train,
                                                   regression_cols=task_info["regression_targets"],
                                                  classification_col=task_info["classification_targets"])

        # -------------------------------------------------
        # Save fitted scalers to disk
        # -------------------------------------------------
        save_scalers(scaler_x, scaler_y, task_name, base_path)

        # -------------------------------------------------
        # Apply scaling to train/val/test sets
        # -------------------------------------------------
        (
            X_train_scaled, X_val_scaled, X_test_scaled,
            y_train_scaled, y_val_scaled, y_test_scaled
        ) = transform_with_scalers(
            X_train, X_val, X_test,
            scaler_x,
            y_train, y_val, y_test,
            scaler_y,
            regression_cols=task_info["regression_targets"],
            classification_col=task_info["classification_targets"]
        )


        # -------------------------------------------------
        # Optional: Apply any custom transformation on scaled features
        # Skip this step for classification tasks
        # -------------------------------------------------
        if task_name != "overloaded_node_classification":
            X_train_scaled = transform(pd.DataFrame(X_train_scaled, columns=features))
            X_val_scaled = transform(pd.DataFrame(X_val_scaled, columns=features))
            X_test_scaled = transform(pd.DataFrame(X_test_scaled, columns=features))

        # -------------------------------------------------
        # Store scaled datasets and scalers in dictionaries
        # -------------------------------------------------
        x_train_dict[task_name] = X_train_scaled
        x_val_dict[task_name] = X_val_scaled
        x_test_dict[task_name] = X_test_scaled
        x_scalers[task_name] = scaler_x

        y_train_dict[task_name] = y_train_scaled
        y_val_dict[task_name] = y_val_scaled
        y_test_dict[task_name] = y_test_scaled
        y_scalers[task_name] = scaler_y

    # -----------------------------------------------------
    # Return all processed splits and scalers
    # -----------------------------------------------------
    return (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
    )


In [ ]:
def preprocessing_splits(X_train, y_train, X_val, y_val, X_test, y_test, tasks, base_path):
    """
    Perform full preprocessing pipeline:
    - Fit MinMax scalers for features and (optionally) targets
    - Save scalers to disk
    - Transform datasets using fitted scalers
    - Return scaled datasets and scalers for all tasks
    """

    # -----------------------------------------------------
    # Initialize dictionaries to store splits and scalers
    # -----------------------------------------------------
    x_train_dict, x_val_dict, x_test_dict = {}, {}, {}
    y_train_dict, y_val_dict, y_test_dict = {}, {}, {}
    x_scalers, y_scalers = {}, {}

    # -----------------------------------------------------
    # Loop through each task for preprocessing
    # -----------------------------------------------------
    for task_name,task_info in tasks.items():
        printmd(f"# {task_name}")
        printmd("---")

        # -------------------------------------------------
        # Check for data leakage
        # -------------------------------------------------
        check_node_type_distribution(X_train,X_val,X_test)
        check_data_leakage(X_train,X_val,X_test)

        # -------------------------------------------------
        # Fit MinMaxScaler
        # Skip target scaling for classification tasks
        # -------------------------------------------------
        printmd("## Scalaing")
        if task_name == "overloaded_node_classification":
            scaler_x, scaler_y = fit_minmax_scaler(X_train)
        else:
            scaler_x, scaler_y = fit_minmax_scaler(X_train, y_train,
                                                   regression_cols=task_info["regression_targets"],
                                                  classification_col=task_info["classification_targets"])

        # -------------------------------------------------
        # Save fitted scalers to disk
        # -------------------------------------------------
        save_scalers(scaler_x, scaler_y, task_name, base_path)

        # -------------------------------------------------
        # Apply scaling to train/val/test sets
        # -------------------------------------------------
        (
            X_train_scaled, X_val_scaled, X_test_scaled,
            y_train_scaled, y_val_scaled, y_test_scaled
        ) = transform_with_scalers(
            X_train, X_val, X_test,
            scaler_x,
            y_train, y_val, y_test,
            scaler_y,
            regression_cols=task_info["regression_targets"],
            classification_col=task_info["classification_targets"]
        )


        # -------------------------------------------------
        # Optional: Apply any custom transformation on scaled features
        # Skip this step for classification tasks
        # -------------------------------------------------
        if task_name != "overloaded_node_classification":
            X_train_scaled = transform(pd.DataFrame(X_train_scaled, columns=features))
            X_val_scaled = transform(pd.DataFrame(X_val_scaled, columns=features))
            X_test_scaled = transform(pd.DataFrame(X_test_scaled, columns=features))

        # -------------------------------------------------
        # Store scaled datasets and scalers in dictionaries
        # -------------------------------------------------
        x_train_dict[task_name] = X_train_scaled
        x_val_dict[task_name] = X_val_scaled
        x_test_dict[task_name] = X_test_scaled
        x_scalers[task_name] = scaler_x

        y_train_dict[task_name] = y_train_scaled
        y_val_dict[task_name] = y_val_scaled
        y_test_dict[task_name] = y_test_scaled
        y_scalers[task_name] = scaler_y

    # -----------------------------------------------------
    # Return all processed splits and scalers
    # -----------------------------------------------------
    return (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
    )


In [ ]:
def prepare_cross_domain_evaluation_data(feature_dataset, target_dataset, tasks, base_path, augmented_df):
    """
    Prepare datasets for cross-domain model evaluation.

    This function preprocesses feature and target data from both domains
    without splitting into train/validation/test sets. The entire dataset
    is scaled and returned for evaluation purposes (e.g., evaluating a
    source model on a target domain, and vice versa).

    Steps:
    - Fit MinMax scalers on the entire dataset (per task)
    - Skip target scaling for classification tasks
    - Save fitted scalers to disk
    - Transform full datasets using fitted scalers
    - Return scaled datasets and scalers for all tasks

    Parameters
    ----------
    feature_dataset : dict
        Dictionary of feature DataFrames for each task.
    target_dataset : dict
        Dictionary of target DataFrames/Series for each task.
    tasks : list
        List of task names to preprocess.
    base_path : str
        Directory path to save fitted scalers.

    Returns
    -------
    x_crossdom_dict : dict
        Dictionary of scaled feature DataFrames for each task.
    y_crossdom_dict : dict
        Dictionary of scaled target arrays for each task.
    x_crossdom_scalers : dict
        Dictionary of fitted feature scalers for each task.
    y_crossdom_scalers : dict
        Dictionary of fitted target scalers for each task.
    """

    # -----------------------------------------------------
    # Initialize dictionaries to store processed data
    # -----------------------------------------------------
    x_crossdom_dict, y_crossdom_dict = {}, {}
    x_crossdom_scalers, y_crossdom_scalers = {}, {}

    # -----------------------------------------------------
    # Loop through each task for preprocessing
    # -----------------------------------------------------
    for task_name, task_info in tasks.items():
        printmd(f"# {task_name}")
        printmd("---")
        # Extract feature and target datasets
        X = feature_dataset[task_name]
        y = target_dataset[task_name]

        X, y = remove_augmented_rows_from_test(X, y, augmented_df, features)
        # -------------------------------------------------
        # Fit MinMaxScaler
        # Skip target scaling for classification tasks
        # -------------------------------------------------
        if task_name == "overloaded_node_classification":
            scaler_x, scaler_y = fit_minmax_scaler(X)
        else:
            scaler_x, scaler_y = fit_minmax_scaler(X, y,
                                                   regression_cols=task_info["regression_targets"],
                                                  classification_col=task_info["classification_targets"])

        # -------------------------------------------------
        # Save fitted scalers to disk
        # -------------------------------------------------
        save_scalers(scaler_x, scaler_y, task_name, base_path)

        # -------------------------------------------------
        # Apply scaling to the full dataset
        # -------------------------------------------------
        X_scaled, y_scaled = transform_with_scalers_crossdom(
            X, scaler_x, y, scaler_y,
            regression_cols=task_info["regression_targets"],
            classification_col=task_info["classification_targets"]
        )


        # -------------------------------------------------
        # Optional: Apply any custom transformation
        # Skip this step for classification tasks
        # -------------------------------------------------
        # if task_name != "overloaded_node_classification":
        X_scaled = transform(pd.DataFrame(X_scaled, columns=features))

        # -------------------------------------------------
        # Store scaled data and scalers
        # -------------------------------------------------
        x_crossdom_dict[task_name] = X_scaled
        y_crossdom_dict[task_name] = y_scaled
        x_crossdom_scalers[task_name] = scaler_x
        y_crossdom_scalers[task_name] = scaler_y

    # -----------------------------------------------------
    # Return all processed datasets and scalers
    # -----------------------------------------------------
    return (
        x_crossdom_dict,
        y_crossdom_dict,
        x_crossdom_scalers,
        y_crossdom_scalers
    )


### Model Building

In [ ]:
def training(prepared_models,x_train_dict, x_val_dict, y_train_dict, y_val_dict, base_path, logs_path):
  adapted_models = {}
  models_history = {}
  train_time = {}

  for task_name in tqdm(prepared_models):
    X_train = x_train_dict[task_name]
    X_val = x_val_dict[task_name]
    y_train = y_train_dict[task_name]
    y_val = y_val_dict[task_name]
    model, history, training_time = train_model(prepared_models[task_name], task_name, X_train, y_train, X_val, y_val,base_path, logs_path)
    adapted_models[task_name] = model
    models_history[task_name] = history
    train_time[task_name] = training_time

  time_df.to_csv(training_time_path)
  return adapted_models, models_history, train_time

In [ ]:
def predict(trained_models, x_test_dict, tasks, num_permutations=6):
    """
    Predict using multitask Janossy models.

    Parameters
    ----------
    trained_models : dict
        Keys = task names, Values = trained Keras models.
    x_test_dict : dict
        Keys = task names, Values = X_test arrays.
    tasks : list
        List of task names to predict.
    num_permutations : int
        Number of permutations for Janossy input.

    Returns
    -------
    all_predictions : dict
        Dictionary of predictions per task. For multitask models, contains:
        {
            'regression': array,
            'classification': array
        }
    """
    all_predictions = {}

    for task_name in tqdm(tasks):
        X_test = x_test_dict[task_name]
        X_test = prepare_janossy_input(X_test, num_permutations=num_permutations)

        model = trained_models[task_name]
        y_pred = model.predict(X_test)

        # Handle multitask output
        if isinstance(y_pred, (list, tuple)) and len(y_pred) == 2:
            # Regression + Classification heads
            all_predictions[task_name] = {
                'regression': y_pred[0],
                'classification': y_pred[1]
            }
        else:
            # Single-output model fallback
            all_predictions[task_name] = y_pred

    return all_predictions


In [ ]:
class JanossyPermutedBatchSequence(Sequence):
    """
    Keras Sequence generator for Janossy pooling.
    Generates `k` permutations per sample in each batch.
    """

    def __init__(self, X, y, batch_size=32, num_permutations=10, shuffle_batch=True):
        self.X = np.array(X)  # shape: (num_samples, sequence_length, input_dim)
        self.y = np.array(y)
        self.batch_size = batch_size
        self.num_permutations = num_permutations
        self.shuffle_batch = shuffle_batch
        self.indexes = np.arange(len(self.X))
        self.seq_len = self.X.shape[1]

        # Precompute all permutations if sequence length is small
        if self.seq_len <= 6:
            self.all_perms = list(permutations(range(self.seq_len)))
        else:
            self.all_perms = None  # fallback to random sampling

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_indexes = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = self.X[batch_indexes]
        y_batch = self.y[batch_indexes]

        # Output shape: (batch_size, num_permutations, sequence_length, input_dim)
        X_batch_permuted = np.zeros((len(batch_indexes), self.num_permutations, self.seq_len, self.X.shape[2]))

        for i, sample in enumerate(X_batch):
            if self.all_perms and len(self.all_perms) >= self.num_permutations:
                selected_perms = random.sample(self.all_perms, self.num_permutations)
            else:
                selected_perms = [np.random.permutation(self.seq_len) for _ in range(self.num_permutations)]

            for j, perm in enumerate(selected_perms):
                X_batch_permuted[i, j] = sample[list(perm)]

        return X_batch_permuted, y_batch

    def on_epoch_end(self):
        if self.shuffle_batch:
            np.random.shuffle(self.indexes)


In [ ]:
def prepare_janossy_input(X, num_permutations=10):
    """
    Transforms test data to Janossy-pooling-compatible input shape:
    (num_samples, num_permutations, sequence_length, input_dim)

    Args:
        X: shape (num_samples, sequence_length, input_dim)
        num_permutations: number of permutations to generate per sample
    Returns:
        X_prepared: shape (num_samples, num_permutations, sequence_length, input_dim)
    """
    X = np.array(X)
    num_samples, seq_len, feat_dim = X.shape

    # Try to precompute all permutations if small enough
    if seq_len <= 6:
        all_perms = list(permutations(range(seq_len)))
    else:
        all_perms = None

    X_prepared = np.zeros((num_samples, num_permutations, seq_len, feat_dim))

    for i in range(num_samples):
        if all_perms and len(all_perms) >= num_permutations:
            selected_perms = random.sample(all_perms, num_permutations)
        else:
            selected_perms = [np.random.permutation(seq_len) for _ in range(num_permutations)]

        for j, perm in enumerate(selected_perms):
            X_prepared[i, j] = X[i, list(perm)]

    return X_prepared


In [ ]:
@register_keras_serializable()
class JanossyPooling(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
@register_keras_serializable()
def masked_mse(y_true, y_pred):
    # Mask samples where ALL regression targets are nan
    sample_mask = ~tf.reduce_all(tf.math.is_nan(y_true), axis=-1)

    # Select only unmasked samples
    y_true_masked = tf.boolean_mask(y_true, sample_mask)
    y_pred_masked = tf.boolean_mask(y_pred, sample_mask)

    return tf.reduce_mean(tf.square(y_true_masked - y_pred_masked))


In [ ]:
@register_keras_serializable()
def masked_mae(y_true, y_pred):
    mask = ~tf.reduce_all(tf.math.is_nan(y_true), axis=-1)
    y_true = tf.boolean_mask(y_true, mask)
    y_pred = tf.boolean_mask(y_pred, mask)
    return tf.reduce_mean(tf.abs(y_true - y_pred))


In [ ]:
@register_keras_serializable()
def r2_score_training(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - ss_res / (ss_tot + K.epsilon())

In [ ]:
@register_keras_serializable()
def masked_r2(y_true, y_pred):
    mask = ~tf.reduce_all(tf.math.is_nan(y_true), axis=-1)
    y_true = tf.boolean_mask(y_true, mask)
    y_pred = tf.boolean_mask(y_pred, mask)

    ss_res = tf.reduce_sum(tf.square(y_true - y_pred))
    ss_tot = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
    return 1 - ss_res / (ss_tot + 1e-7)


In [ ]:
def train_model(model, task_name, X_train, y_train, X_val, y_val,base_path,logs_path):
    # input_dim = X_train.shape[1] if task_name.startswith('overloaded') else X_train.shape[2]
    # input_dim = X_train.shape[2]
    # output_dim = y_train.shape[1] if len(y_train.shape) > 1 else 1  # Handle multi-output

    # Clone model (NO side effects)
    model = clone_model_with_weights(model)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    # Split regression and classification targets
    y_val = np.array(y_val)
    y_train_reg, y_train_cls = y_train[:, :-1], y_train[:, -1]
    y_val_reg, y_val_cls = y_val[:, :-1], y_val[:, -1]

    input_dim = X_train.shape[2]
    reg_output_dim = y_train_reg.shape[1]
    num_permutations = 6


    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

    # Create the logs directory if it doesn't exist
    os.makedirs(logs_path, exist_ok=True)
    csv_logger = CSVLogger(logs_path + f'/{task_name}_logs.csv', append=False)


    # Training time tracking
    start_time = time.time()

    # Prepare the generator
    # train_sequence = JanossyPermutedBatchSequence( X_train, y_train, batch_size=32, num_permutations=num_permutations,shuffle_batch=True)
    X_train = prepare_janossy_input(X_train, num_permutations=num_permutations)
    X_val = prepare_janossy_input(X_val, num_permutations=num_permutations)

    history = model.fit(
        X_train,
        {'regression_output': y_train_reg, 'classification_output': y_train_cls},
        validation_data=(X_val, {'regression_output': y_val_reg, 'classification_output': y_val_cls}),
        epochs=40,
        batch_size=32,
        callbacks=[early_stopping, lr_scheduler, csv_logger],
        verbose=1
    )

    training_time = time.time() - start_time
    # target_model_name = task_name + "_" + model_type + base_path.split("/")[-1]
    target_model_name = f"{task_name}_{os.path.basename(base_path)}"
    time_df.loc[target_model_name, MODEL_NAME] = training_time
    print(f"Training time for {target_model_name}: {training_time:.2f} seconds")

    # Save the model
    # path = f'{base_path}/{task_name}/{model_type}'
    os.makedirs(base_path, exist_ok=True)
    model.save(os.path.join(base_path, f"{task_name}.keras"))

    return model, history, training_time


## Features and Targets List

In [ ]:
X_train = pd.read_csv(os.path.join(PATH_TO_SPLITS+"target/fold_1/", "X_train.csv"))
y_train = pd.read_csv(os.path.join(PATH_TO_SPLITS+"target/fold_1/", "y_train.csv"))

# 🎯 Define target columns
target_prefixes = ("cpu_usage_", "ram_usage_", "overloaded_node", "medium_latency")
targets = [
    col for col in y_train.columns
    if col.startswith(target_prefixes) and "idle" not in col
]

# 🔢 Numerical & Categorical Features
categorical_features = ["node_type"]
numerical_features = [col for col in X_train.columns if col.startswith("rate_")]
features = categorical_features + numerical_features

# 🧭 Target Categorization
categorical_targets = ["overloaded_node"]
numerical_targets = [col for col in targets if col not in categorical_targets]


In [ ]:
# Dataframe division by features and target
tasks = {"Multi_Task_regression": {"features": features, "targets": numerical_targets},
         "Multi_Task_classification": {"features": features, "targets": categorical_targets} }

In [ ]:
tasks_unified = {"Multi_Task": {"features": features, "targets": numerical_targets+categorical_targets,
                                "regression_targets": numerical_targets, "classification_targets":categorical_targets} }

# Load and Preprocessing CV Splits: Train–Test-Validation Feature Scaling

## Methods

In [ ]:
def filter_by_step_size(
    X_train,
    y_train,
    step_size,
    rate_columns=[
        'rate_function_curl',
        'rate_function_eat_memory',
        'rate_function_env',
        'rate_function_figlet',
        'rate_function_nmap',
        'rate_function_shasum'
    ],
    mode="multiple",
    verbose=True
):
    """
    Flexible rate-based filtering for DFaaS training data.

    Parameters
    ----------
    step_size : int or tuple/list of length 2
        Single step or two steps.
    mode : str
        One of:
        - "multiple"  : divisible by step_size
        - "and"       : divisible by both steps
        - "or"        : divisible by either step
        - "lt"        : less than first step
        - "gt"        : greater than first step
        - "between"   : between steps (inclusive)
        - "outside"   : less than low OR greater than high
    """

    # ======================
    # BEFORE FILTERING
    # ======================
    n_before = len(X_train)
    n_unique_before = X_train.drop_duplicates().shape[0]
    dup_ratio_before = 1.0 - (n_unique_before / n_before)

    rates = X_train[rate_columns]

    # ======================
    # BUILD MASK
    # ======================
    if isinstance(step_size, (int, np.integer)):
        if step_size <= 0:
            raise ValueError("step_size must be positive")

        mask = (rates % step_size == 0).all(axis=1)

    elif isinstance(step_size, (list, tuple)) and len(step_size) == 2:
        s1, s2 = step_size

        if mode == "and":
            mask = ((rates % s1 == 0) & (rates % s2 == 0)).all(axis=1)

        elif mode == "or":
            mask = ((rates % s1 == 0) | (rates % s2 == 0)).all(axis=1)

        elif mode == "lt":
            mask = (rates < s1).all(axis=1)

        elif mode == "gt":
            mask = (rates > s1).all(axis=1)

        elif mode == "between":
            low, high = min(s1, s2), max(s1, s2)
            mask = ((rates >= low) & (rates <= high)).all(axis=1)

        elif mode == "outside":
          low, high = min(s1, s2), max(s1, s2)

          mask_low = ((rates == 0) | (rates <= low)).all(axis=1)
          mask_high = ((rates == 0) | (rates >= high)).all(axis=1)

          mask = mask_low | mask_high


        else:
            raise ValueError(
                "Invalid mode. Use: multiple, and, or, lt, gt, between, outside"
            )

    else:
        raise ValueError("step_size must be int or tuple/list of length 2")

    # ======================
    # APPLY FILTERING
    # ======================
    X_train_filtered = X_train.loc[mask].reset_index(drop=True)
    y_train_filtered = y_train.loc[mask].reset_index(drop=True)

    # ======================
    # AFTER FILTERING
    # ======================
    n_after = len(X_train_filtered)
    n_unique_after = X_train_filtered.drop_duplicates().shape[0]
    dup_ratio_after = (
        1.0 - (n_unique_after / n_after) if n_after > 0 else 0.0
    )
    retention_ratio = n_after / n_before if n_before > 0 else 0.0

    # ======================
    # LOGGING
    # ======================
    if verbose:
        print("=" * 70)
        print(f"Filtering summary | step={step_size}, mode={mode}")
        print("-" * 70)
        print(f"Samples before : {n_before}")
        print(f"Samples after  : {n_after}")
        print(f"Retention      : {retention_ratio:.3f}")

        print("\nFeature uniqueness:")
        print(f"  Before : {n_unique_before} ({dup_ratio_before:.2%} dup)")
        print(f"  After  : {n_unique_after} ({dup_ratio_after:.2%} dup)")

        print("\nRate coverage (after filtering):")
        for col in rate_columns:
            uniq = np.sort(X_train_filtered[col].unique())
            print(
                f"  {col}: "
                f"min={uniq.min() if len(uniq) else 'NA'}, "
                f"max={uniq.max() if len(uniq) else 'NA'}, "
                f"unique={len(uniq)}"
            )

        if n_unique_after <= 1:
            print("\n⚠ WARNING: Feature space collapsed.")
        elif dup_ratio_after > 0.9:
            print("\n⚠ WARNING: High duplicate ratio (>90%).")

        print("=" * 70)

    return X_train_filtered, y_train_filtered


In [ ]:
def print_split_stats(fold_idx, X_train, X_val, X_test, y_train, y_val, y_test):
    printmd(f"### ===== Fold {fold_idx} statistics =====")

    print("X shapes:")
    print(f"  Train: {X_train.shape}")
    print(f"  Val  : {X_val.shape}")
    print(f"  Test : {X_test.shape}")

    print("y shapes:")
    print(f"  Train: {y_train.shape}")
    print(f"  Val  : {y_val.shape}")
    print(f"  Test : {y_test.shape}")

    print("=====================================")


In [ ]:
def load_xy_split(split_dir):

    X_train = pd.read_csv(os.path.join(split_dir, "X_train.csv"))
    y_train = pd.read_csv(os.path.join(split_dir, "y_train.csv"))

    X_val = pd.read_csv(os.path.join(split_dir, "X_val.csv"))
    y_val = pd.read_csv(os.path.join(split_dir, "y_val.csv"))

    X_test = pd.read_csv(os.path.join(split_dir, "X_test.csv"))
    y_test = pd.read_csv(os.path.join(split_dir, "y_test.csv"))

    return X_train, y_train, X_val, y_val, X_test, y_test


In [ ]:
def load_all_folds(base_split_path, scaler_path, tasks, k=5):
    """
    Load X/y train, val, test splits for all folds and preprocess them.

    Parameters
    ----------
    base_split_path : str
        Path like 'data/splits/source' or 'data/splits/target'
    scaler_path : str
        Base path to store scalers per fold
    tasks : list
        Task definitions for preprocessing
    k : int
        Number of folds

    Returns
    -------
    folds : dict
        fold_idx -> tuple of processed splits
    """

    folds = {}
    training_samples = {}
    training_unique_samples = {}
    filtered_training_samples = {}
    filtered_training_unique_samples = {}

    printmd(f"### Loading {k}-fold splits from: {base_split_path}")

    for fold_idx in range(1, k + 1):
        printmd(f"# --- Processing fold {fold_idx} ---")

        fold_path = os.path.join(base_split_path, f"fold_{fold_idx}")
        fold_scaler_path = os.path.join(scaler_path, f"fold_{fold_idx}")

        # Load raw splits
        X_train, y_train, X_val, y_val, X_test, y_test = load_xy_split(fold_path)

        # Print raw stats (before preprocessing)
        print_split_stats(
            fold_idx,
            X_train, X_val, X_test,
            y_train, y_val, y_test
        )

        training_samples[fold_idx] = len(X_train)
        training_unique_samples[fold_idx] = len(X_train.drop_duplicates())

        X_train, y_train = filter_by_step_size(
            X_train, y_train,
            step_size=STEP,
            mode=MODE,
        )
        filtered_training_samples[fold_idx] = len(X_train)
        filtered_training_unique_samples[fold_idx] = len(X_train.drop_duplicates())

        printmd("### Running preprocessing...")
        (
        x_train_dict, x_val_dict, x_test_dict, x_scalers,
        y_train_dict, y_val_dict, y_test_dict, y_scalers
                  )   = preprocessing_splits(
                          X_train, y_train,
                          X_val, y_val,
                          X_test, y_test,
                          tasks,
                          fold_scaler_path
                      )

        printmd("### Preprocessing completed.")

        # Store processed outputs (assumes preprocessing_splits populates these)
        folds[fold_idx] = (x_train_dict, x_val_dict, x_test_dict,
        y_train_dict, y_val_dict, y_test_dict)

        printmd(f"### Fold {fold_idx} loaded and stored successfully.")

    printmd("### All folds loaded successfully.")
    return folds, training_samples, training_unique_samples, filtered_training_samples, filtered_training_unique_samples


## Load Target Data Splits

In [ ]:
(folds_target, training_samples, training_unique_samples,
 filtered_training_samples, filtered_training_unique_samples) = load_all_folds(base_split_path = PATH_TO_SPLITS+"target/",
                              scaler_path = PATH_TO_SCALERS + 'target/', tasks=tasks_unified, k=5)

### Loading 5-fold splits from: /content/drive/MyDrive/system_metrics_predictions/data/splits/V2/target/

# --- Processing fold 1 ---

### ===== Fold 1 statistics =====

X shapes:
  Train: (76634, 7)
  Val  : (10741, 7)
  Test : (13419, 7)
y shapes:
  Train: (76634, 23)
  Val  : (10741, 23)
  Test : (13419, 23)
Filtering summary | step=40, mode=multiple
----------------------------------------------------------------------
Samples before : 76634
Samples after  : 5299
Retention      : 0.069

Feature uniqueness:
  Before : 20530 (73.21% dup)
  After  : 1116 (78.94% dup)

Rate coverage (after filtering):
  rate_function_curl: min=0.0, max=200.0, unique=6
  rate_function_eat_memory: min=0.0, max=120.0, unique=4
  rate_function_env: min=0.0, max=200.0, unique=6
  rate_function_figlet: min=0.0, max=200.0, unique=6
  rate_function_nmap: min=0.0, max=200.0, unique=6
  rate_function_shasum: min=0.0, max=200.0, unique=6


### Running preprocessing...

...

## Scalaing

[INFO] MinMaxScaler fitted:
  - Features shape: (5354, 7)
  - Regression targets shape: (5354, 22)
  - Classification column '['overloaded_node']' excluded from scaling

[INFO] Scalers saved for target 'Multi_Task'

[INFO] Transformation complete.
  - X_train shape: (5354, 7)
  - X_val shape: (10721, 7)
  - X_test shape: (13623, 7)



### Preprocessing completed.

### Fold 5 loaded and stored successfully.

### All folds loaded successfully.

In [ ]:
training_samples

{1: 76634, 2: 75940, 3: 76006, 4: 75796, 5: 75858}

In [ ]:
training_unique_samples

{1: 20530, 2: 20508, 3: 20498, 4: 20526, 5: 20476}

In [ ]:
filtered_training_samples

{1: 5299, 2: 5514, 3: 5286, 4: 5147, 5: 5354}

In [ ]:
filtered_training_unique_samples

{1: 1116, 2: 1135, 3: 1120, 4: 1112, 5: 1158}

In [ ]:
samples_values = list(training_samples.values())
unique_samples_values = list(training_unique_samples.values())
filtered_samples_values = list(filtered_training_samples.values())
filtered_unique_samples_values = list(filtered_training_unique_samples.values())

mean_samples = np.mean(samples_values)
std_samples = np.std(samples_values)

mean_unique_samples = np.mean(unique_samples_values)
std_unique_samples = np.std(unique_samples_values)

mean_filtered_samples = np.mean(filtered_samples_values)
std_filtered_samples = np.std(filtered_samples_values)

mean_filtered_unique_samples = np.mean(filtered_unique_samples_values)
std_filtered_unique_samples = np.std(filtered_unique_samples_values)



print(f"Mean number of training samples across folds: {mean_samples:.0f}")
print(f"Standard deviation of training samples across folds: {std_samples:.0f}")

print(f"Mean number of unique training samples across folds: {mean_unique_samples:.0f}")
print(f"Standard deviation of unique training samples across folds: {std_unique_samples:.0f}")

print(f"Mean number of filtered training samples across folds: {mean_filtered_samples:.0f}")
print(f"Standard deviation of filtered  training samples across folds: {std_filtered_samples:.0f}")

print(f"Mean number of filtered unique  training samples across folds: {mean_filtered_unique_samples:.0f}")
print(f"Standard deviation of filtered  unique  training samples across folds: {std_filtered_unique_samples:.0f}")

Mean number of training samples across folds: 76047
Standard deviation of training samples across folds: 302
Mean number of unique training samples across folds: 20508
Standard deviation of unique training samples across folds: 20
Mean number of filtered training samples across folds: 5320
Standard deviation of filtered  training samples across folds: 119
Mean number of filtered unique  training samples across folds: 1128
Standard deviation of filtered  unique  training samples across folds: 17


# Load Models

### Methods

In [ ]:

def load_models_from_directory(base_path):
    """
    Load all Keras models from a given directory into a dictionary.

    Parameters
    ----------
    base_path : str
        Path to the directory containing saved model files (.keras or .h5).

    Returns
    -------
    models_dict : dict
        Dictionary where keys are model names (file names without extension),
        and values are the loaded Keras models.
    """

    models_dict = {}

    # Check if directory exists
    if not os.path.exists(base_path):
        print(f"[ERROR] Directory does not exist: {base_path}")
        return models_dict
    if not os.path.isdir(base_path):
        print(f"[ERROR] Path is not a directory: {base_path}")
        return models_dict

    # Iterate only over files in the given directory
    for file in os.listdir(base_path):
        if file.endswith(".keras") or file.endswith(".h5"):
            model_path = os.path.join(base_path, file)
            model_name = os.path.splitext(file)[0]  # filename without extension
            # Register as replacement for <lambda>

            try:
                models_dict[model_name] = keras.models.load_model(model_path)
                print(f"[INFO] Loaded model: {model_name} from {model_path}")
            except Exception as e:
                print(f"[WARNING] Could not load {model_path}: {e}")

    if not models_dict:
        print(f"[INFO] No models found in directory: {base_path}")

    return models_dict

In [ ]:
@register_keras_serializable()
class JanossyPooling(layers.Layer):
    def call(self, inputs):
        return tf.reduce_mean(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
@register_keras_serializable()
def r2_score_training(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - ss_res / (ss_tot + K.epsilon())

In [ ]:
def print_models_summary(model_dict):
    """
    Print summaries for multiple Keras models stored in a dictionary.

    Parameters
    ----------
    model_dict : dict
        Dictionary where keys are task names (str)
        and values are compiled or loaded Keras models.
    """
    if not model_dict:
        print("[WARNING] No models found in the dictionary.")
        return

    for task_name, model in model_dict.items():
        printmd("---")
        printmd(f"# [MODEL SUMMARY] Task: {task_name}")
        printmd("---")

        if model is None:
            print(f"[ERROR] No model found for task '{task_name}'.")
        else:
            try:
                model.summary(line_length=100)
            except Exception as e:
                print(f"[ERROR] Could not print summary for '{task_name}': {e}")

        printmd("---")


### Source

In [ ]:
source_path = PATH_TO_LOAD_MODELS + "source/fold_5"

In [ ]:
loaded_models_dict_source = load_models_from_directory(source_path)

[INFO] Loaded model: Multi_Task from /content/drive/MyDrive/system_metrics_predictions/output/system-forecaster-models/baselines/V1/source/fold_5/Multi_Task.keras


In [ ]:
print_models_summary(loaded_models_dict_source)

---

# [MODEL SUMMARY] Task: Multi_Task

---

Model: "janossy_multitask_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)    │ (None, 6, None, 8)      │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ masking_layer               │ (None, 6, None, 8)      │              0 │ input_layer[0][0]       │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ time_distributed_embedding  │ (None, 6, None, 64)     │            576 │ masking_layer[0][0]     │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ time_distributed_rnn        │ (None, 6, 80)           │         35,040 │ time_distributed_embed… │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ janossy_pooling             │ (None, 80)              │              0 │ time_distributed_rnn[0… │
│ (JanossyPooling)            │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ dense_final (Dense)         │ (None, 128)             │         10,368 │ janossy_pooling[0][0]   │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ dropout_final (Dropout)     │ (None, 128)             │              0 │ dense_final[0][0]       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_dense (Dense)    │ (None, 128)             │         16,512 │ dropout_final[0][0]     │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ classification_dense        │ (None, 128)             │         16,512 │ dropout_final[0][0]     │
│ (Dense)                     │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_dropout          │ (None, 128)             │              0 │ regression_dense[0][0]  │
│ (Dropout)                   │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ classification_dropout      │ (None, 128)             │              0 │ classification_dense[0… │
│ (Dropout)                   │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_output (Dense)   │ (None, 22)              │          2,838 │ regression_dropout[0][… │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ classification_output       │ (None, 1)               │            129 │ classification_dropout… │
│ (Dense)                     │                         │                │                         │
└─────────────────────────────┴─────────────────────────┴────────────────┴─────────────────────────┘

 Total params: 245,927 (960.66 KB)

 Trainable params: 81,975 (320.21 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 163,952 (640.44 KB)

---

# K-fold Before Adaptation Results


## Methods

In [ ]:
def aggregate_kfold_results(all_results, base_path=None):
    """
    Accepts a list of per-fold results (dicts with DataFrames for classification and regression)
    and returns a dict with average and std of metrics for each task.
    Rows remain in original order, columns are interleaved as <metric>_mean, <metric>_std.
    Optionally saves the results to an Excel file.
    """

    def process_task(dfs, index_col):
        dfs_copy = [df.copy() for df in dfs]
        numeric_cols = dfs_copy[0].select_dtypes(include="number").columns

        # Compute mean and std
        mean_df = pd.concat(dfs_copy).groupby(index_col)[numeric_cols].mean()
        std_df = pd.concat(dfs_copy).groupby(index_col)[numeric_cols].std()

        # Interleave columns: mean then std
        interleaved_cols = []
        interleaved_data = pd.DataFrame(index=mean_df.index)
        for col in numeric_cols:
            interleaved_data[col + "_mean"] = mean_df[col]
            interleaved_data[col + "_std"] = std_df[col]
            interleaved_cols.extend([col + "_mean", col + "_std"])

        # Keep original row order
        original_order = dfs_copy[0][index_col]
        interleaved_data = interleaved_data.loc[original_order]

        return interleaved_data[interleaved_cols]

    cls_summary = process_task([fold["Multi_Task_classification"] for fold in all_results], "class")
    reg_summary = process_task([fold["Multi_Task_regression"] for fold in all_results], "Target")

    aggregated_results = {
        "Multi_Task_classification": cls_summary,
        "Multi_Task_regression": reg_summary
    }

    if base_path:
        os.makedirs(base_path, exist_ok=True)
        for task_name, df in aggregated_results.items():
            filename = f"{task_name}_aggregated.xlsx"
            file_path = os.path.join(base_path, filename)
            df.to_excel(file_path, index=True) # index=True to save the 'class' or 'Target' column
            printmd(f"#### [INFO] Saved aggregated results for {task_name} to {file_path}")

    return aggregated_results

In [ ]:
def aggregate_kfold_results_with_error(all_results, base_path=None):
    """
    Aggregates K-Fold results and returns a dict of DataFrames where each metric
    column contains mean ± std as a string. Optionally saves the results to an Excel file.
    """

    def process_task(dfs, index_col):
        dfs_copy = [df.copy() for df in dfs]
        numeric_cols = dfs_copy[0].select_dtypes(include="number").columns

        # Compute mean and std
        mean_df = pd.concat(dfs_copy).groupby(index_col)[numeric_cols].mean()
        std_df = pd.concat(dfs_copy).groupby(index_col)[numeric_cols].std()

        # Combine mean ± std
        combined_df = pd.DataFrame(index=mean_df.index)
        for col in numeric_cols:
            combined_df[col] = mean_df[col].round(6).astype(str) + " ± " + std_df[col].round(6).astype(str)

        # Keep original row order
        original_order = dfs_copy[0][index_col]
        combined_df = combined_df.loc[original_order]

        return combined_df

    cls_summary = process_task([fold["Multi_Task_classification"] for fold in all_results], "class")
    reg_summary = process_task([fold["Multi_Task_regression"] for fold in all_results], "Target")

    aggregated_results = {
        "Multi_Task_classification": cls_summary,
        "Multi_Task_regression": reg_summary
    }

    if base_path:
        os.makedirs(base_path, exist_ok=True)
        for task_name, df in aggregated_results.items():
            filename = f"{task_name}_aggregated_with_error.xlsx"
            file_path = os.path.join(base_path, filename)
            df.to_excel(file_path, index=True) # index=True to save the 'class' or 'Target' column
            printmd(f"#### [INFO] Saved aggregated results for {task_name} (with error) to {file_path}")

    return aggregated_results

In [ ]:
def eval_before_adaptation_results(folds, loaded_models_dict, tasks, tasks_unified,results_path,plot_path):
  all_results = []
  for fold_idx, (x_train_dict, x_val_dict, x_test_dict, y_train_dict, y_val_dict, y_test_dict) in folds.items():

    printmd(f"# Fold {fold_idx} Before Adaptation Results:")

    all_predictions = predict(loaded_models_dict, x_test_dict, tasks_unified)

    results_df = show_result(all_predictions,y_test_dict,tasks,base_path= results_path + f"/fold_{fold_idx}",
                             plot_path=plot_path+ f"/fold_{fold_idx}")

    all_results.append(results_df)

  return all_results


## Source Model - Target Data

In [ ]:
results_df_source_target_before_adaptation = eval_before_adaptation_results(folds_target,loaded_models_dict_source,
                                                                        tasks,tasks_unified,
                                                                        PATH_TO_RESULTS+"source_target/before_adaptation",
                                                                        PATH_TO_PLOTS+"source_target/before_adaptation")

Output hidden; open in https://colab.research.google.com to view.

## Display Foldwise Before Adaptation Results

### Source Model - Target Data

In [ ]:
for fold, results in enumerate(results_df_source_target_before_adaptation, start=1):
  printmd(f"# Fold: {fold}")
  printmd("---")
  display_results(results)

# Fold: 1

---

# Multi_Task_regression

---

,Target,MSE,RMSE,MAPE,R2,Std Dev
0,overall_model_performance,0.018030,0.134276,6.059078,0.080411,
1,cpu_usage_function_shasum,0.015992,0.126458,7.965152,0.288556,0.205717
2,ram_usage_function_shasum,0.018518,0.136081,8.709411,-0.515024,0.111393
3,medium_latency_function_shasum,0.008341,0.091330,1.994185,-0.144376,0.058902
4,cpu_usage_function_env,0.003820,0.061802,3.537931,0.866105,0.168505
5,ram_usage_function_env,0.007216,0.084949,4.473171,-0.067283,0.09565
6,medium_latency_function_env,0.003430,0.058565,0.954229,0.168374,0.04025
7,cpu_usage_function_eat_memory,0.004467,0.066835,2.268048,0.865424,0.173236
8,ram_usage_function_eat_memory,0.012888,0.113526,3.892159,0.532163,0.198567
9,medium_latency_function_eat_memory,0.011567,0.107552,3.656481,-0.466440,0.137499


...

---

,Target,MSE,RMSE,MAPE,R2,Std Dev
0,overall_model_performance,0.016894,0.129978,5.708975,0.179840,
1,cpu_usage_function_shasum,0.013579,0.116531,7.181258,0.423849,0.20588
2,ram_usage_function_shasum,0.018137,0.134673,8.482896,-0.477714,0.11397
3,medium_latency_function_shasum,0.010941,0.104599,1.967940,0.046478,0.057288
4,cpu_usage_function_env,0.003257,0.057068,3.280117,0.901336,0.168819
5,ram_usage_function_env,0.013242,0.115075,5.862076,0.191376,0.092053
6,medium_latency_function_env,0.002692,0.051884,0.879876,0.078745,0.037514
7,cpu_usage_function_eat_memory,0.003887,0.062347,2.201470,0.864595,0.133481
8,ram_usage_function_eat_memory,0.006407,0.080043,3.099566,0.751651,0.150114
9,medium_latency_function_eat_memory,0.005596,0.074809,2.275115,0.291241,0.101675


# Multi_Task_classification

---

,class,precision,recall,f1-score,support
0,0.0,0.974531,0.913658,0.943113,12103.000000
1,1.0,0.540861,0.809868,0.648577,1520.000000
2,accuracy,0.902077,0.902077,0.902077,0.902077
3,macro avg,0.757696,0.861763,0.795845,13623.000000
4,weighted avg,0.926144,0.902077,0.910250,13623.000000


## Display Aggregated Before Adaptation Result

### Source Model - Target Data

In [ ]:
cv_results_source_target_before_adaptation = aggregate_kfold_results(results_df_source_target_before_adaptation,
                                                                 base_path=PATH_TO_RESULTS+"source_target/before_adaptation/aggregate")

#### [INFO] Saved aggregated results for Multi_Task_classification to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/before_adaptation/aggregate/Multi_Task_classification_aggregated.xlsx

#### [INFO] Saved aggregated results for Multi_Task_regression to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/before_adaptation/aggregate/Multi_Task_regression_aggregated.xlsx

In [ ]:
display_results(cv_results_source_target_before_adaptation)

# Multi_Task_classification

---

,precision_mean,precision_std,recall_mean,recall_std,f1-score_mean,f1-score_std,support_mean,support_std
class,,,,,,,,
0.0,0.972428,0.002282,0.898958,0.018923,0.934165,0.010220,11896.600000,147.031629
1.0,0.525422,0.033699,0.812536,0.015276,0.637385,0.021786,1618.000000,59.485292
accuracy,0.888668,0.015568,0.888668,0.015568,0.888668,0.015568,0.888668,0.015568
macro avg,0.748925,0.016572,0.855747,0.006260,0.785775,0.015899,13514.600000,93.665896
weighted avg,0.918839,0.006007,0.888668,0.015568,0.898607,0.012664,13514.600000,93.665896


# Multi_Task_regression

---

,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
Target,,,,,,,,
overall_model_performance,0.017100,0.000656,0.130748,0.002506,5.771540,0.175126,0.136149,0.037153
cpu_usage_function_shasum,0.015436,0.001042,0.124183,0.004295,7.764514,0.327967,0.299047,0.072161
ram_usage_function_shasum,0.014823,0.003244,0.121178,0.013179,7.493145,1.018432,-0.804027,0.302954
medium_latency_function_shasum,0.008801,0.001467,0.093559,0.007700,1.912969,0.073542,0.002791,0.090583
cpu_usage_function_env,0.003246,0.000354,0.056909,0.003055,3.335217,0.122000,0.890419,0.014830
ram_usage_function_env,0.012589,0.005843,0.109783,0.025908,5.554091,1.111514,0.086009,0.142631
medium_latency_function_env,0.002903,0.000610,0.053639,0.005732,0.897973,0.042804,0.022850,0.129416
cpu_usage_function_eat_memory,0.004013,0.000267,0.063320,0.002070,2.215745,0.042059,0.872456,0.006910
ram_usage_function_eat_memory,0.008956,0.002413,0.093995,0.012317,3.441951,0.284877,0.662994,0.081468


In [ ]:
cv_results_source_target_before_adaptation = aggregate_kfold_results_with_error(results_df_source_target_before_adaptation,
                                                                 base_path=PATH_TO_RESULTS+"source_target/before_adaptation/aggregate")

#### [INFO] Saved aggregated results for Multi_Task_classification (with error) to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/before_adaptation/aggregate/Multi_Task_classification_aggregated_with_error.xlsx

#### [INFO] Saved aggregated results for Multi_Task_regression (with error) to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/before_adaptation/aggregate/Multi_Task_regression_aggregated_with_error.xlsx

In [ ]:
display_results(cv_results_source_target_before_adaptation)

# Multi_Task_classification

---

,precision,recall,f1-score,support
class,,,,
0.0,0.972428 ± 0.002282,0.898958 ± 0.018923,0.934165 ± 0.01022,11896.6 ± 147.031629
1.0,0.525422 ± 0.033699,0.812536 ± 0.015276,0.637385 ± 0.021786,1618.0 ± 59.485292
accuracy,0.888668 ± 0.015568,0.888668 ± 0.015568,0.888668 ± 0.015568,0.888668 ± 0.015568
macro avg,0.748925 ± 0.016572,0.855747 ± 0.00626,0.785775 ± 0.015899,13514.6 ± 93.665896
weighted avg,0.918839 ± 0.006007,0.888668 ± 0.015568,0.898607 ± 0.012664,13514.6 ± 93.665896


# Multi_Task_regression

---

,MSE,RMSE,MAPE,R2
Target,,,,
overall_model_performance,0.0171 ± 0.000656,0.130748 ± 0.002506,5.77154 ± 0.175126,0.136149 ± 0.037153
cpu_usage_function_shasum,0.015436 ± 0.001042,0.124183 ± 0.004295,7.764514 ± 0.327967,0.299047 ± 0.072161
ram_usage_function_shasum,0.014823 ± 0.003244,0.121178 ± 0.013179,7.493145 ± 1.018432,-0.804027 ± 0.302954
medium_latency_function_shasum,0.008801 ± 0.001467,0.093559 ± 0.0077,1.912969 ± 0.073542,0.002791 ± 0.090583
cpu_usage_function_env,0.003246 ± 0.000354,0.056909 ± 0.003055,3.335217 ± 0.122,0.890419 ± 0.01483
ram_usage_function_env,0.012589 ± 0.005843,0.109783 ± 0.025908,5.554091 ± 1.111514,0.086009 ± 0.142631
medium_latency_function_env,0.002903 ± 0.00061,0.053639 ± 0.005732,0.897973 ± 0.042804,0.02285 ± 0.129416
cpu_usage_function_eat_memory,0.004013 ± 0.000267,0.06332 ± 0.00207,2.215745 ± 0.042059,0.872456 ± 0.00691
ram_usage_function_eat_memory,0.008956 ± 0.002413,0.093995 ± 0.012317,3.441951 ± 0.284877,0.662994 ± 0.081468


# Preparing Models for Domain Adaptation by Freezing Layers

## Method

In [ ]:
def print_trainable_layers(model_dict):
    """
    Print the trainable status of layers for each model in a dictionary.

    Parameters
    ----------
    model_dict : dict
        Dictionary where keys are model names (e.g., task names)
        and values are Keras models.
    """
    if not model_dict:
        print("[WARNING] No models found in the dictionary.")
        return

    for name, model in model_dict.items():
        printmd("---")
        printmd(f"# [MODEL LAYERS] {name}")
        printmd("---")

        try:
            for layer in model.layers:
                printmd(f"*  Layer name: {layer.name} Trainable: **{layer.trainable}**")
        except Exception as e:
            print(f"[ERROR] Could not access layers for model '{name}': {e}")

        printmd("---")


In [ ]:
def compile_for_finetuning(model, reg_loss_function=masked_mse, cls_loss_function='binary_crossentropy',lr=1e-4):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss={
            "regression_output": reg_loss_function,
            "classification_output": cls_loss_function
        },
        loss_weights={
            'regression_output': 5.0,
            'classification_output': 1.0
        },
        metrics={
            'regression_output': [masked_mae, masked_r2],
            'classification_output': ['accuracy']
        }
    )
    return model


In [ ]:
def prepare_models_for_projection_adaptation(
    loaded_models,
    domain="target",
    proj_dim=128,
    projection_type="linear",
    insert_after="dropout_final",
    lr=1e-3,
    bottleneck_dim=64
):
    """
    Prepare a set of loaded models for domain adaptation by:
    - Replacing the projection layer
    - Freezing the entire model
    - Training ONLY the new projection layer

    Parameters
    ----------
    loaded_models : dict
        Dictionary where keys are task names and values are pre-trained Keras models.

    domain : str
        Target domain name.

    proj_dim : int
        Output dimension of the projection layer.

    projection_type : str, default="linear"
        Type of projection ("linear", "nonlinear", etc.)

    old_proj_layer_name : str
        Name of the old projection layer to be replaced.

    lr : float, default=1e-3
        Learning rate for projection fine-tuning.

    Returns
    -------
    prepared_models : dict
        Dictionary with models ready for projection-based domain adaptation.
    """

    if not loaded_models:
        print("[WARNING] No models provided.")
        return {}

    prepared_models = {}

    for task_name, model in loaded_models.items():
        printmd("---")
        printmd(f"# [PROJECTION ADAPTATION] Task: {task_name}")
        printmd("---")

        # Replace projection & train only projection
        prepared_model = add_task_specific_projection_before_output(
            model=model,
            domain=domain,
            proj_dim=proj_dim,
            projection_type=projection_type,
            bottleneck_dim=bottleneck_dim,
        )

        # Compile for projection-only training
        prepared_model = compile_for_finetuning(
            prepared_model, lr=lr
        )

        prepared_models[task_name] = prepared_model

        print(f"[INFO] Model '{task_name}' ready for PROJECTION adaptation.\n")

    return prepared_models


## Source Model - Adapted on Target data

In [ ]:
prepared_models_source_proj = prepare_models_for_projection_adaptation(loaded_models_dict_source,domain="target",
                                                                        proj_dim=128,
                                                                        projection_type=PROJECTION,
                                                                        lr=1e-3,bottleneck_dim=64)

---

# [PROJECTION ADAPTATION] Task: Multi_Task

---

[INFO] Model 'Multi_Task' ready for PROJECTION adaptation.



In [ ]:
print_models_summary(prepared_models_source_proj)

---

# [MODEL SUMMARY] Task: Multi_Task

---

Model: "janossy_multitask_model_proj_before_out_target"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)    │ (None, 6, None, 8)      │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ masking_layer               │ (None, 6, None, 8)      │              0 │ input_layer[0][0]       │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ time_distributed_embedding  │ (None, 6, None, 64)     │            576 │ masking_layer[0][0]     │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ time_distributed_rnn        │ (None, 6, 80)           │         35,040 │ time_distributed_embed… │
│ (TimeDistributed)           │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ janossy_pooling             │ (None, 80)              │              0 │ time_distributed_rnn[0… │
│ (JanossyPooling)            │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ dense_final (Dense)         │ (None, 128)             │         10,368 │ janossy_pooling[0][0]   │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ dropout_final (Dropout)     │ (None, 128)             │              0 │ dense_final[0][0]       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_dense (Dense)    │ (None, 128)             │         16,512 │ dropout_final[0][0]     │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ classification_dense        │ (None, 128)             │         16,512 │ dropout_final[0][0]     │
│ (Dense)                     │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_dropout          │ (None, 128)             │              0 │ regression_dense[0][0]  │
│ (Dropout)                   │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ classification_dropout      │ (None, 128)             │              0 │ classification_dense[0… │
│ (Dropout)                   │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ adapter_layer_target_regre… │ (None, 128)             │         16,576 │ regression_dropout[0][… │
│ (AdapterLayer)              │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ adapter_layer_target_class… │ (None, 128)             │         16,576 │ classification_dropout… │
│ (AdapterLayer)              │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ regression_output (Dense)   │ (None, 22)              │          2,838 │ adapter_layer_target_r… │
├─────────────────────────────┼─────────────────────────┼────

 Total params: 115,127 (449.71 KB)

 Trainable params: 33,152 (129.50 KB)

 Non-trainable params: 81,975 (320.21 KB)

---

In [ ]:
print_trainable_layers(prepared_models_source_proj)

---

# [MODEL LAYERS] Multi_Task

---

*  Layer name: input_layer Trainable: **False**

*  Layer name: masking_layer Trainable: **False**

...

*  Layer name: adapter_layer_target_regression Trainable: **True**

*  Layer name: adapter_layer_target_classification Trainable: **True**

*  Layer name: regression_output Trainable: **False**

*  Layer name: classification_output Trainable: **False**

---

# K-Fold Domain Adaptation

## Methods

In [ ]:
metric_map_loss = {'Multi_Target_regression': "loss",
 'overloaded_node_classification': "loss"}

In [ ]:
metric_map_r2 = {'Multi_Target_regression': "r2_score_training",
 'overloaded_node_classification': "accuracy"}

In [ ]:
def cross_validate_domain_adaptation(prepared_models,folds, tasks,tasks_unified,
                                     models_path,logs_path,results_path,plot_path):
  all_results = []
  training_times = {}

  for fold_idx, (x_train_dict, x_val_dict, x_test_dict, y_train_dict, y_val_dict, y_test_dict) in folds.items():
    printmd(f"# ===== Fold {fold_idx} =====")

    print(f"Start Training {fold_idx}: {datetime.datetime.now()}")
    adapted_models, models_history, training_time = training(prepared_models, x_train_dict, x_val_dict,
                                              y_train_dict, y_val_dict,
                                              base_path= models_path + f"/fold_{fold_idx}",
                                              logs_path = logs_path + f"/fold_{fold_idx}")

    print(f"End Training {fold_idx}: {datetime.datetime.now()}")

    plot_training_histories_separate(models_history,metric_map=metric_map_loss,save_dir = plot_path + f"/fold_{fold_idx}")
    plot_training_histories_separate(models_history,metric_map=metric_map_r2, save_dir = plot_path + f"/fold_{fold_idx}")

    all_predictions = predict(adapted_models, x_test_dict, tasks_unified)

    printmd(f"## Fold {fold_idx} After Adaptation Results:")
    results_df = show_result(all_predictions,y_test_dict,tasks,base_path= results_path + f"/fold_{fold_idx}",
                             plot_path=plot_path + f"/fold_{fold_idx}")

    all_results.append(results_df)
    training_times[fold_idx] = training_time

  return all_results, training_times

## Domain Adaptation Cross Validation

In [ ]:
start = datetime.datetime.now()
print(f"Start: {start}")

Start: 2026-01-28 18:28:58.742671


In [ ]:
all_results_df_source_target, all_training_times_source_target =  cross_validate_domain_adaptation(prepared_models_source_proj,folds_target,tasks,tasks_unified,
                                 models_path= PATH_TO_MODELS + f"source_target/after_adaptation/",
                                 logs_path = PATH_TO_TRAINING_LOGS + f"source_target/after_adaptation/",
                                 results_path = PATH_TO_RESULTS + f"source_target/after_adaptation/",
                                plot_path = PATH_TO_PLOTS + f"source_target/after_adaptation/")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
end = datetime.datetime.now()
print(f"End: {end}")
print(f"Duration: {end-start}")

End: 2026-01-28 18:43:39.217556
Duration: 0:14:40.474885


In [ ]:
all_training_times_source_target

{1: {'Multi_Task': 69.67319059371948},
 2: {'Multi_Task': 117.53363132476807},
 3: {'Multi_Task': 108.98162174224854},
 4: {'Multi_Task': 113.91898369789124},
 5: {'Multi_Task': 146.3338532447815}}

In [ ]:
{i[0]: i[1]["Multi_Task"] for i in all_training_times_source_target.items()}

{1: 69.67319059371948,
 2: 117.53363132476807,
 3: 108.98162174224854,
 4: 113.91898369789124,
 5: 146.3338532447815}

## Display Foldwise Domain Adaptation Results

### Source Model - Target Data

In [ ]:
for fold, results in enumerate(all_results_df_source_target, start=1):
  printmd(f"# Fold: {fold}")
  printmd("---")
  display_results(results)

# Fold: 1

---

# Multi_Task_regression

---

,Target,MSE,RMSE,MAPE,R2,Std Dev
0,overall_model_performance,0.006857,0.082809,3.563328,0.621422,
1,cpu_usage_function_shasum,0.002318,0.048147,2.700463,0.896871,0.134188
2,ram_usage_function_shasum,0.006710,0.081913,4.732461,0.451056,0.072585
3,medium_latency_function_shasum,0.004513,0.067178,1.990902,0.380862,0.041875
4,cpu_usage_function_env,0.002371,0.048688,2.819923,0.916901,0.162244
5,ram_usage_function_env,0.002353,0.048504,3.068884,0.652049,0.060118
6,medium_latency_function_env,0.003675,0.060624,1.425766,0.108857,0.019817
7,cpu_usage_function_eat_memory,0.002518,0.050177,2.352984,0.924147,0.17174
8,ram_usage_function_eat_memory,0.006821,0.082591,3.196292,0.752391,0.131946
9,medium_latency_function_eat_memory,0.004273,0.065365,2.323296,0.458353,0.063031


...

---

,Target,MSE,RMSE,MAPE,R2,Std Dev
0,overall_model_performance,0.006394,0.079962,3.437835,0.659039,
1,cpu_usage_function_shasum,0.002096,0.045782,2.551089,0.911073,0.140141
2,ram_usage_function_shasum,0.006246,0.079029,4.578127,0.491140,0.076968
3,medium_latency_function_shasum,0.006485,0.080532,2.131005,0.434791,0.051397
4,cpu_usage_function_env,0.002216,0.047076,2.870959,0.932861,0.169527
5,ram_usage_function_env,0.004904,0.070027,4.082940,0.700558,0.096833
6,medium_latency_function_env,0.002561,0.050603,1.335344,0.123688,0.021085
7,cpu_usage_function_eat_memory,0.001734,0.041644,2.555610,0.939590,0.165747
8,ram_usage_function_eat_memory,0.005516,0.074267,3.145565,0.786197,0.121687
9,medium_latency_function_eat_memory,0.003692,0.060762,2.119649,0.532418,0.052754


# Multi_Task_classification

---

,class,precision,recall,f1-score,support
0,0.0,0.954359,0.981327,0.967655,12103.000000
1,1.0,0.808149,0.626316,0.705708,1520.000000
2,accuracy,0.941716,0.941716,0.941716,0.941716
3,macro avg,0.881254,0.803821,0.836682,13623.000000
4,weighted avg,0.938046,0.941716,0.938428,13623.000000


## Display Aggregated Domain Adaptation Result

### Source Model - Target Data

In [ ]:
cv_results_source_target_adaptation = aggregate_kfold_results(all_results_df_source_target,
                                            base_path=PATH_TO_RESULTS+f"source_target/after_adaptation/aggregate/")

#### [INFO] Saved aggregated results for Multi_Task_classification to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/after_adaptation/aggregate/Multi_Task_classification_aggregated.xlsx

#### [INFO] Saved aggregated results for Multi_Task_regression to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/after_adaptation/aggregate/Multi_Task_regression_aggregated.xlsx

In [ ]:
display_results(cv_results_source_target_adaptation)

# Multi_Task_classification

---

,precision_mean,precision_std,recall_mean,recall_std,f1-score_mean,f1-score_std,support_mean,support_std
class,,,,,,,,
0.0,0.938538,0.012943,0.977597,0.003453,0.957636,0.007389,11896.600000,147.031629
1.0,0.761274,0.035382,0.530936,0.087946,0.623439,0.072011,1618.000000,59.485292
accuracy,0.923855,0.013364,0.923855,0.013364,0.923855,0.013364,0.923855,0.013364
macro avg,0.849906,0.023594,0.754267,0.044266,0.790538,0.039580,13514.600000,93.665896
weighted avg,0.917218,0.016028,0.923855,0.013364,0.917412,0.016425,13514.600000,93.665896


# Multi_Task_regression

---

,MSE_mean,MSE_std,RMSE_mean,RMSE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
Target,,,,,,,,
overall_model_performance,0.006543,0.000208,0.080883,0.001279,3.490992,0.045289,0.642678,0.013999
cpu_usage_function_shasum,0.002144,0.000185,0.046268,0.001999,2.659083,0.160787,0.902945,0.007837
ram_usage_function_shasum,0.004655,0.001672,0.067382,0.011996,4.082022,0.526725,0.456500,0.022841
medium_latency_function_shasum,0.005450,0.001036,0.073566,0.006936,2.028612,0.064412,0.385657,0.033875
cpu_usage_function_env,0.002223,0.000146,0.047123,0.001560,2.864971,0.150523,0.924979,0.007456
ram_usage_function_env,0.004764,0.002392,0.067131,0.017923,3.810382,0.570613,0.665449,0.030550
medium_latency_function_env,0.002747,0.000770,0.051999,0.007326,1.445479,0.077997,0.093496,0.028843
cpu_usage_function_eat_memory,0.001995,0.000309,0.044564,0.003345,2.408051,0.095753,0.936798,0.007382
ram_usage_function_eat_memory,0.006371,0.000500,0.079767,0.003202,3.117923,0.068549,0.759306,0.015430


In [ ]:
cv_results_source_target_adaptation = aggregate_kfold_results_with_error(all_results_df_source_target,
                                                    base_path=PATH_TO_RESULTS+f"source_target/after_adaptation/aggregate")

#### [INFO] Saved aggregated results for Multi_Task_classification (with error) to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/after_adaptation/aggregate/Multi_Task_classification_aggregated_with_error.xlsx

#### [INFO] Saved aggregated results for Multi_Task_regression (with error) to /content/drive/MyDrive/system_metrics_predictions/output/results/multi_task_domain_adaptation/data_reduction_experiment/V3/source_target/after_adaptation/aggregate/Multi_Task_regression_aggregated_with_error.xlsx

In [ ]:
display_results(cv_results_source_target_adaptation)

# Multi_Task_classification

---

,precision,recall,f1-score,support
class,,,,
0.0,0.938538 ± 0.012943,0.977597 ± 0.003453,0.957636 ± 0.007389,11896.6 ± 147.031629
1.0,0.761274 ± 0.035382,0.530936 ± 0.087946,0.623439 ± 0.072011,1618.0 ± 59.485292
accuracy,0.923855 ± 0.013364,0.923855 ± 0.013364,0.923855 ± 0.013364,0.923855 ± 0.013364
macro avg,0.849906 ± 0.023594,0.754267 ± 0.044266,0.790538 ± 0.03958,13514.6 ± 93.665896
weighted avg,0.917218 ± 0.016028,0.923855 ± 0.013364,0.917412 ± 0.016425,13514.6 ± 93.665896


# Multi_Task_regression

---

,MSE,RMSE,MAPE,R2
Target,,,,
overall_model_performance,0.006543 ± 0.000208,0.080883 ± 0.001279,3.490992 ± 0.045289,0.642678 ± 0.013999
cpu_usage_function_shasum,0.002144 ± 0.000185,0.046268 ± 0.001999,2.659083 ± 0.160787,0.902945 ± 0.007837
ram_usage_function_shasum,0.004655 ± 0.001672,0.067382 ± 0.011996,4.082022 ± 0.526725,0.4565 ± 0.022841
medium_latency_function_shasum,0.00545 ± 0.001036,0.073566 ± 0.006936,2.028612 ± 0.064412,0.385657 ± 0.033875
cpu_usage_function_env,0.002223 ± 0.000146,0.047123 ± 0.00156,2.864971 ± 0.150523,0.924979 ± 0.007456
ram_usage_function_env,0.004764 ± 0.002392,0.067131 ± 0.017923,3.810382 ± 0.570613,0.665449 ± 0.03055
medium_latency_function_env,0.002747 ± 0.00077,0.051999 ± 0.007326,1.445479 ± 0.077997,0.093496 ± 0.028843
cpu_usage_function_eat_memory,0.001995 ± 0.000309,0.044564 ± 0.003345,2.408051 ± 0.095753,0.936798 ± 0.007382
ram_usage_function_eat_memory,0.006371 ± 0.0005,0.079767 ± 0.003202,3.117923 ± 0.068549,0.759306 ± 0.01543


## Clear Memory

In [ ]:
del all_results_df_source_target
del cv_results_source_target_adaptation
del prepared_models_source_proj
del INSERT_SOURCE
gc.collect()

253443